# Tabular Profiler — Stage 1 of 2

**What this does:** profiles a tabular competition dataset and prints **one long text
report** (also saved to `report.md` + `facts.json`). You paste that report plus the
contest description into a chatbot, and the chatbot writes the whole solution.

**How to use at the contest**
1. Edit **only** the CONFIG cell below (usually just `TRAIN_PATH` / `TEST_PATH`).
2. Run All.
3. Scroll to the last code cell, copy everything between the `COPY EVERYTHING BETWEEN
   THE LINES` markers, and paste it into the Stage-2 prompt in the final markdown cell.

Every section is individually guarded: if one fails it is logged into the report as a
`SECTION FAILED` entry and the notebook keeps going. It never stops halfway.

Tabular data only (csv / tsv / parquet / feather / xlsx). Use `image_profiler.ipynb`
for image datasets.

In [ ]:
# =====================================================================================
# CONFIG — the only cell you should need to edit
# =====================================================================================

# --- Required: where the data lives -------------------------------------------------
TRAIN_PATH = "/kaggle/input/competition/train.csv"   # csv/tsv/parquet/feather/xlsx
TEST_PATH  = "/kaggle/input/competition/test.csv"    # None if the contest has no test file
SAMPLE_SUBMISSION_PATH = None                        # None = auto-detect next to train

# --- Column roles: leave None to auto-detect ----------------------------------------
TARGET    = None    # str, or list of str for multi-target. None = auto-detect
ID_COL    = None    # str. None = auto-detect
TASK      = None    # "binary" | "multiclass" | "regression". None = auto-detect
GROUP_COL = None    # str. Set if rows share an entity -> forces GroupKFold
TIME_COL  = None    # str. Set for time-series -> forces expanding-window CV

# --- Budgets (tuned for a 2-hour contest) -------------------------------------------
MAX_ROWS   = 300_000   # heavy analyses sample down to this many rows
N_FOLDS    = 5         # CV folds for the baseline
BASELINE_TIME_BUDGET_S = 120   # stop adding folds past this; a partial CV still reports
MAX_COLS_IN_TABLE      = 150   # column table shows this many, then "... N more"
MAX_LEAK_FEATURES      = 60    # cap single-feature leak probes

# --- Which heavy sections to run ----------------------------------------------------
RUN_ADVERSARIAL = True   # train-vs-test shift detection
RUN_LEAK_CHECK  = True   # single-feature leakage probes
RUN_BASELINE    = True   # quick CV baseline score
RUN_FIGURES     = True   # save/show plots
PARALLEL_HEAVY  = True   # overlap the 3 heavy sections so CPU and GPU both stay busy

# --- Compute: GPU is used ONLY where it actually saves wall-clock --------------------
# "auto" detects Kaggle (2 GPUs available) vs Colab (1 GPU) vs local.
PLATFORM  = "auto"     # "auto" | "kaggle" | "colab" | "local"
GPU_MODE  = "auto"     # "auto" = use GPU only when the data is big enough to win
                       # "off"  = never      | "force" = always when a GPU exists
GPU_MIN_CELLS = 2_000_000   # rows x features below this -> CPU is faster (transfer
                            # + kernel-launch overhead dominates on small frames)

# --- Output --------------------------------------------------------------------------
SEED    = 42
OUT_DIR = "profiler_out"

# --- Automated-test override: harmless at the contest, leave it alone ---------------
import json as _json
import os as _os
_ov = _os.environ.get("PROFILER_CONFIG_JSON")
if _ov:
    _d = _json.loads(_ov)
    globals().update(_d)
    print("[config] overrides applied:", ", ".join(sorted(_d)))

In [ ]:
# =====================================================================================
# INFRASTRUCTURE — report buffer, section guard, formatting helpers
# =====================================================================================
import gc
import json
import math
import os
import platform
import random
import re
import sys
import time
import traceback
import warnings
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)

OUT = Path(OUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

T_START = time.perf_counter()
SECTIONS = []        # ordered list of section dicts -> assembled into the report
_CUR = None          # section currently collecting say() output
FLAGS = {}           # flag text -> severity (dict keeps insertion order + dedupes)
FACTS = {}           # machine-readable mirror of the key findings


def say(*parts):
    """Print a line and append it to the current section of the report."""
    text = " ".join(str(p) for p in parts) if parts else ""
    print(text)
    (_CUR["lines"] if _CUR is not None else SECTIONS[0]["lines"]).append(text)


def flag(severity, text):
    """Record one AUTO-FLAG. Deduplicated; highest severity seen wins."""
    rank = {"high": 3, "medium": 2, "low": 1}
    if text in FLAGS and rank[FLAGS[text]] >= rank[severity]:
        return
    FLAGS[text] = severity


@contextmanager
def section(key, title):
    """Run an analysis section; log any exception into the report and continue."""
    global _CUR
    sec = {"key": key, "title": title, "lines": [], "error": None, "seconds": 0.0}
    SECTIONS.append(sec)
    _CUR = sec
    t0 = time.perf_counter()
    print(f"\n{'=' * 84}\n{title}\n{'=' * 84}")
    try:
        yield sec
    except Exception as exc:                                   # never propagate
        sec["error"] = f"{type(exc).__name__}: {exc}"
        say(f"**SECTION FAILED** — `{type(exc).__name__}`: {exc}")
        say("```")
        for line in traceback.format_exc().strip().splitlines()[-7:]:
            say(line)
        say("```")
        flag("high", f"Section '{title}' failed ({type(exc).__name__}: {exc}). "
                     f"Its facts are missing from this report — do not rely on them.")
    finally:
        sec["seconds"] = time.perf_counter() - t0
        _CUR = None
        print(f"\n[{sec['seconds']:.2f}s] {title}")


def need(*names):
    """Fail this section early (and legibly) if an earlier section did not produce X."""
    missing = [n for n in names if globals().get(n, None) is None]
    if missing:
        raise RuntimeError(f"prerequisite(s) unavailable: {missing} "
                           f"— an earlier section must have failed")


# --- formatting ----------------------------------------------------------------------

def fmt(v, nd=4):
    """Compact, stable formatting for report cells."""
    if v is None:
        return "-"
    if isinstance(v, (bool, np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        f = float(v)
        if not math.isfinite(f):
            return "nan" if math.isnan(f) else ("inf" if f > 0 else "-inf")
        if f != 0 and (abs(f) >= 1e7 or abs(f) < 1e-4):
            return f"{f:.3e}"
        return f"{round(f, nd):,}".rstrip("0").rstrip(".") if "." in f"{round(f, nd)}" \
            else f"{round(f, nd):,}"
    s = str(v).replace("|", "\\|").replace("\n", " ")
    return s if len(s) <= 80 else s[:77] + "..."


def md_table(headers, rows, cap=None, unit="rows"):
    """Render a markdown pipe table, capped with a '... N more' line."""
    rows = list(rows)
    shown = rows if cap is None or len(rows) <= cap else rows[:cap]
    out = ["| " + " | ".join(str(h) for h in headers) + " |",
           "|" + "|".join("---" for _ in headers) + "|"]
    for r in shown:
        out.append("| " + " | ".join(fmt(v) for v in r) + " |")
    if len(rows) > len(shown):
        out.append(f"... {len(rows) - len(shown)} more {unit} not shown")
    return "\n".join(out)


def say_table(headers, rows, cap=None, unit="rows"):
    if not list(rows):
        say("_(none)_")
        return
    say(md_table(headers, rows, cap=cap, unit=unit))


def cap_items(items, cap=50):
    items = list(items)
    if len(items) <= cap:
        return ", ".join(str(i) for i in items)
    return ", ".join(str(i) for i in items[:cap]) + f"  ... +{len(items) - cap} more"


def human_bytes(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f}{unit}"
        n /= 1024


# --- pandas 2.x / 3.x compatible dtype probes ---------------------------------------
# pandas 3 stores text as a "str" dtype, not object, so `dtype == object` is unreliable.

def is_text_dtype(s):
    return bool(pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s))


def is_num_dtype(s):
    return bool(pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s))


def is_dt_dtype(s):
    return bool(pd.api.types.is_datetime64_any_dtype(s))


def to_num(s):
    """Coerce to float ndarray with NaN for non-numeric — safe for both pandas majors."""
    return pd.to_numeric(s, errors="coerce").astype("float64").to_numpy()


def parse_dt(s):
    """Parse to naive datetime64. tz-aware input is normalised to UTC then stripped
    (mixing tz-aware and naive timestamps raises in every later comparison)."""
    d = pd.to_datetime(s, errors="coerce", format="mixed")
    if is_dt_dtype(d) and getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert("UTC").dt.tz_localize(None)
    return d


def jsonable(o):
    """Recursively convert numpy/pandas scalars so json.dump never chokes."""
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        f = float(o)
        return f if math.isfinite(f) else None
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (np.ndarray, pd.Index, pd.Series)):
        return jsonable(list(o))
    if isinstance(o, (pd.Timestamp,)):
        return str(o)
    if o is None or isinstance(o, (str, int, float, bool)):
        if isinstance(o, float) and not math.isfinite(o):
            return None
        return o
    return str(o)


def opt_import(name):
    """Import an optional dependency, returning None instead of raising."""
    try:
        return __import__(name)
    except Exception:
        return None


def pkg_version(name):
    try:
        import importlib.metadata as md
        return md.version(name)
    except Exception:
        mod = sys.modules.get(name)
        return getattr(mod, "__version__", None) if mod else None


print(f"infrastructure ready | out={OUT.resolve()} | pandas {pd.__version__} | "
      f"numpy {np.__version__}")

In [ ]:
# =====================================================================================
# 1. ENVIRONMENT & COMPUTE PLAN
# =====================================================================================
COMPUTE = None

with section("env", "1. ENVIRONMENT & COMPUTE PLAN"):
    say(f"- Python: `{sys.version.split()[0]}` on `{platform.platform()}`")
    say(f"- Executable: `{sys.executable}`")

    cpu_count = os.cpu_count() or 2
    psutil = opt_import("psutil")
    ram_gb = disk_gb = None
    if psutil:
        ram_gb = psutil.virtual_memory().total / 1e9
        try:
            disk_gb = psutil.disk_usage(str(OUT.resolve())).free / 1e9
        except Exception:
            pass
    say(f"- CPU cores: **{cpu_count}**"
        + (f" | RAM: **{ram_gb:.1f} GB**" if ram_gb else "")
        + (f" | free disk: {disk_gb:.1f} GB" if disk_gb else ""))

    # platform detection
    plat = PLATFORM
    if plat == "auto":
        if Path("/kaggle/input").exists() or _os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            plat = "kaggle"
        elif "google.colab" in sys.modules or Path("/content").exists():
            plat = "colab"
        else:
            plat = "local"
    say(f"- Detected platform: **{plat}**")

    # GPU inventory (torch if present, else nvidia-smi, else none)
    gpus = []
    torch = opt_import("torch")
    if torch is not None and getattr(torch, "cuda", None) and torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            gpus.append({"index": i, "name": p.name, "vram_gb": p.total_memory / 1e9})
    if not gpus:
        try:
            import subprocess
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=20)
            for i, line in enumerate(out.stdout.strip().splitlines()):
                if line.strip():
                    nm, mem = (line.split(",") + [""])[:2]
                    gpus.append({"index": i, "name": nm.strip(), "vram_gb": mem.strip()})
        except Exception:
            pass

    if gpus:
        for g in gpus:
            vram = g["vram_gb"]
            say(f"- GPU {g['index']}: **{g['name']}** "
                f"({vram:.1f} GB)" if isinstance(vram, float) else
                f"- GPU {g['index']}: **{g['name']}** ({vram})")
    else:
        say("- GPU: **none detected** (CPU-only run)")
        flag("low", "No GPU is available, so generated solution code must train on CPU "
                    "(prefer GBDTs over deep models and keep the model small).")

    # how many GPUs we are allowed to use: Kaggle grants 2, Colab 1
    gpu_budget = min(len(gpus), 2 if plat == "kaggle" else 1)

    libs = ["scikit-learn", "lightgbm", "xgboost", "catboost", "optuna", "shap",
            "category_encoders", "polars", "pyarrow", "scipy", "matplotlib",
            "statsmodels", "torch"]
    rows, missing = [], []
    for name in libs:
        mod_name = {"scikit-learn": "sklearn", "category_encoders": "category_encoders"}\
            .get(name, name)
        mod = opt_import(mod_name)
        ver = pkg_version(name) or (getattr(mod, "__version__", None) if mod else None)
        rows.append([name, "yes" if mod else "NO", ver or "-"])
        if not mod:
            missing.append(name)
    say("")
    say("**Installed libraries** (generated code may use only the `yes` rows):")
    say_table(["library", "installed", "version"], rows)
    if missing:
        say("")
        say(f"- Missing: `{cap_items(missing)}`")

    # internet probe — pip installs are impossible in most offline contests
    internet = False
    try:
        import socket
        socket.setdefaulttimeout(4)
        socket.create_connection(("pypi.org", 443)).close()
        internet = True
    except Exception:
        internet = False
    say(f"- Internet (pypi.org:443 reachable): **{internet}**")
    if not internet:
        flag("medium", "No internet: generated code must not pip-install anything or "
                       "download pretrained weights; use only the installed libraries.")

    # --- decide where the baseline trains ------------------------------------------
    # GPU only pays off on big frames. pip-installed LightGBM is CPU-only, so for GPU we
    # prefer XGBoost (its wheels ship CUDA support); LightGBM stays the CPU choice.
    COMPUTE = {
        "platform": plat,
        "cpu_count": cpu_count,
        "ram_gb": ram_gb,
        "n_gpus_present": len(gpus),
        "gpu_budget": gpu_budget,
        "gpu_names": [g["name"] for g in gpus],
        "internet": internet,
        "missing_libs": missing,
        "gpu_mode": GPU_MODE,
        "use_gpu": False,          # finalized in section 2 once the shape is known
        "gpu_reason": "not decided yet (needs data shape)",
        "cpu_workers": max(1, cpu_count - 1),
    }
    say("")
    say(f"- Compute plan: up to **{gpu_budget} GPU(s)** usable on this platform, "
        f"**{COMPUTE['cpu_workers']}** CPU worker threads for the parallel stages.")
    say("- GPU-vs-CPU decision for the baseline is made in section 2 "
        f"(threshold: rows x features >= {GPU_MIN_CELLS:,} cells).")
    FACTS["environment"] = {k: v for k, v in COMPUTE.items()}
    FACTS["environment"]["python"] = sys.version.split()[0]
    FACTS["environment"]["pandas"] = pd.__version__

In [ ]:
# =====================================================================================
# 2. LOAD & OVERVIEW
# =====================================================================================
DF = DF_TEST = SUB = None
DF_SAMPLE = None
SUB_PATH_USED = None

SUB_NAME_HINTS = ("sample_submission", "samplesubmission", "sample_sub", "submission")


def read_table(path, nrows=None):
    """Read csv/tsv/parquet/feather/xlsx, with a separator fallback for 1-column csv."""
    p = Path(path)
    suf = p.suffix.lower()
    if suf in (".parquet", ".pq"):
        return pd.read_parquet(p)
    if suf == ".feather":
        return pd.read_feather(p)
    if suf in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(p)
    if suf == ".tsv":
        return pd.read_csv(p, sep="\t", nrows=nrows)
    if suf in (".gz", ".zip", ".bz2", ".xz") or suf in (".csv", ".txt", ""):
        df = pd.read_csv(p, nrows=nrows)
        if df.shape[1] == 1:                      # probably not comma-separated
            for sep in (";", "\t", "|"):
                try:
                    alt = pd.read_csv(p, sep=sep, nrows=nrows)
                except Exception:
                    continue
                if alt.shape[1] > df.shape[1]:
                    return alt
        return df
    return pd.read_csv(p, sep=None, engine="python", nrows=nrows)


def reader_call(path):
    """The exact pandas call that reproduces read_table — quoted in CODE-WRITER FACTS."""
    suf = Path(path).suffix.lower()
    return {".parquet": "pd.read_parquet", ".pq": "pd.read_parquet",
            ".feather": "pd.read_feather", ".xlsx": "pd.read_excel",
            ".xls": "pd.read_excel", ".tsv": 'pd.read_csv(..., sep="\\t")'}\
        .get(suf, "pd.read_csv")


def find_sample_submission(train_path):
    """Look for a submission template beside train (and one directory up)."""
    base = Path(train_path).parent
    cands = []
    for d in (base, base.parent):
        if not d.exists():
            continue
        for f in d.iterdir():
            if f.is_file() and any(h in f.name.lower() for h in SUB_NAME_HINTS):
                cands.append(f)
    cands.sort(key=lambda f: (0 if "sample" in f.name.lower() else 1, len(f.name)))
    return cands[0] if cands else None


with section("load", "2. LOAD & OVERVIEW"):
    tp = Path(TRAIN_PATH)
    if not tp.exists():
        raise FileNotFoundError(f"TRAIN_PATH does not exist: {tp.resolve()}")
    DF = read_table(tp)
    say(f"- **train**: `{tp.resolve()}`")
    say(f"  - shape **{DF.shape[0]:,} rows x {DF.shape[1]} cols**, "
        f"in-memory {human_bytes(DF.memory_usage(deep=True).sum())}, "
        f"read with `{reader_call(tp)}`")

    if TEST_PATH:
        tep = Path(TEST_PATH)
        if tep.exists():
            DF_TEST = read_table(tep)
            say(f"- **test**: `{tep.resolve()}`")
            say(f"  - shape **{DF_TEST.shape[0]:,} rows x {DF_TEST.shape[1]} cols**, "
                f"read with `{reader_call(tep)}`")
        else:
            say(f"- **test**: path set but MISSING -> `{tep.resolve()}`")
            flag("high", f"TEST_PATH does not exist ({tep}); treat this as a "
                         f"train-only task or correct the path before generating code.")
    else:
        say("- **test**: not configured (train-only task)")
        flag("medium", "No test file: hold out a validation split yourself and make the "
                       "submission step optional in the generated code.")

    sp = Path(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH else \
        find_sample_submission(tp)
    if sp and Path(sp).exists():
        try:
            SUB = read_table(sp)
            SUB_PATH_USED = str(Path(sp).resolve())
            say(f"- **sample submission**: `{SUB_PATH_USED}` "
                f"-> {SUB.shape[0]:,} rows x {SUB.shape[1]} cols, "
                f"columns `{list(SUB.columns)}`")
        except Exception as exc:
            say(f"- sample submission found at `{sp}` but unreadable: {exc}")
    else:
        say("- **sample submission**: not found "
            "(searched for *sample_submission* next to train and one level up)")
        flag("medium", "No sample_submission file was found; infer the submission "
                       "format from the contest description and state the assumption.")

    # dtype census
    kinds = {}
    for c in DF.columns:
        k = str(DF[c].dtype)
        kinds[k] = kinds.get(k, 0) + 1
    say("")
    say("**Train dtype counts:** " + ", ".join(f"`{k}` x{v}" for k, v in
                                               sorted(kinds.items())))

    # heavy-analysis sample
    if DF.shape[0] > MAX_ROWS:
        DF_SAMPLE = DF.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
        say(f"- Heavy analyses use a random sample of **{MAX_ROWS:,}** rows "
            f"(full data has {DF.shape[0]:,}).")
        flag("low", f"Train has {DF.shape[0]:,} rows; this report's heavy stats come "
                    f"from a {MAX_ROWS:,}-row sample, but train the final model on all "
                    f"rows.")
    else:
        DF_SAMPLE = DF
        say("- Heavy analyses use the full train set (under MAX_ROWS).")

    # finalize the GPU decision now that the shape is known
    cells = int(DF.shape[0]) * int(DF.shape[1])
    if COMPUTE["gpu_budget"] == 0 or GPU_MODE == "off":
        COMPUTE["use_gpu"] = False
        COMPUTE["gpu_reason"] = ("GPU_MODE='off'" if GPU_MODE == "off"
                                 else "no GPU detected")
    elif GPU_MODE == "force":
        COMPUTE["use_gpu"] = True
        COMPUTE["gpu_reason"] = "GPU_MODE='force'"
    else:
        COMPUTE["use_gpu"] = cells >= GPU_MIN_CELLS
        COMPUTE["gpu_reason"] = (
            f"{cells:,} cells >= threshold {GPU_MIN_CELLS:,}" if COMPUTE["use_gpu"]
            else f"only {cells:,} cells (< {GPU_MIN_CELLS:,}): CPU is faster here "
                 f"because host-to-device transfer dominates")
    say(f"- **Baseline device: {'GPU' if COMPUTE['use_gpu'] else 'CPU'}** "
        f"({COMPUTE['gpu_reason']}).")

    # full column table
    say("")
    say(f"**Column table** (train; showing up to {MAX_COLS_IN_TABLE}):")
    rows = []
    n = len(DF)
    for c in DF.columns:
        s = DF[c]
        nn = int(s.notna().sum())
        ex = s.dropna()
        ex = fmt(ex.iloc[0]) if len(ex) else "-"
        rows.append([c, str(s.dtype), int(s.nunique(dropna=True)),
                     f"{100.0 * (n - nn) / max(n, 1):.2f}", ex])
    say_table(["column", "dtype", "nunique", "null_%", "example"], rows,
              cap=MAX_COLS_IN_TABLE, unit="columns")

    if DF_TEST is not None:
        only_tr = [c for c in DF.columns if c not in DF_TEST.columns]
        only_te = [c for c in DF_TEST.columns if c not in DF.columns]
        say("")
        say(f"- Columns in train but not test ({len(only_tr)}): "
            f"`{cap_items(only_tr) or 'none'}`")
        say(f"- Columns in test but not train ({len(only_te)}): "
            f"`{cap_items(only_te) or 'none'}`")

    FACTS["data"] = {
        "train_path": str(tp.resolve()),
        "train_rows": int(DF.shape[0]), "train_cols": int(DF.shape[1]),
        "test_path": str(Path(TEST_PATH).resolve()) if (TEST_PATH and
                                                        Path(TEST_PATH).exists()) else None,
        "test_rows": int(DF_TEST.shape[0]) if DF_TEST is not None else None,
        "sample_submission_path": SUB_PATH_USED,
        "sample_submission_columns": list(SUB.columns) if SUB is not None else None,
        "train_reader": reader_call(tp),
        "all_columns": list(map(str, DF.columns)),
    }

In [ ]:
# =====================================================================================
# 3. SCHEMA INFERENCE — target, id, task, per-column types
# =====================================================================================
TARGET_NAME = None
TARGET_LIST = []
ID_NAME = None
TASK_NAME = None
COLTYPES = {}
CLASS_MAP = None
REF_CLASS = None     # class whose rate the "mean target by X" tables measure
METRIC_GUESS = None

TARGET_NAME_HINTS = ("target", "label", "y", "class", "outcome", "is_", "_flag",
                     "survived", "price", "sales", "score", "rating", "churn", "default",
                     "loss", "claim", "demand", "revenue", "sold", "win", "fraud")
# Exact names first; token matches only count as weak evidence, and only on whole
# word-ish tokens -- substring matching made every "num_*" measurement look like an id.
ID_EXACT_NAMES = ("id", "index", "idx", "row_id", "rowid", "uid", "uuid", "guid", "key",
                  "pk", "record_id", "sample_id")
ID_TOKENS = ("id", "ids", "uuid", "guid", "key", "index", "idx", "no", "num", "code",
             "serial")


def name_tokens(name):
    """Split a column name into lowercase word tokens ('num_skewed' -> {num, skewed})."""
    return {t for t in re.split(r"[^0-9a-z]+", str(name).lower()) if t}


def looks_datetime(s, min_ok=0.8):
    """True if most non-null string values parse as dates."""
    vals = s.dropna().astype(str)
    if len(vals) == 0:
        return False
    probe = vals.sample(min(300, len(vals)), random_state=SEED)
    # a bare integer column is an id/count far more often than a date
    if probe.str.fullmatch(r"-?\d{1,9}").mean() > 0.5:
        return False
    parsed = pd.to_datetime(probe, errors="coerce", format="mixed")
    return float(parsed.notna().mean()) >= min_ok


def infer_type(s, n_rows, name=""):
    """Classify one column into the vocabulary used throughout the report."""
    nu = int(s.nunique(dropna=True))
    lname = str(name).lower()
    if nu <= 1:
        return "constant"
    if pd.api.types.is_bool_dtype(s):
        return "bool"
    if is_dt_dtype(s):
        return "datetime"
    if is_num_dtype(s):
        if nu == n_rows and n_rows > 20 and (
                any(h in lname for h in ("id", "key", "uuid", "index"))
                or bool(s.dropna().is_monotonic_increasing)):
            return "id_like"
        if nu == 2:
            return "binary_num"
        if nu <= 20:
            return "num_lowcard"
        return "numeric"
    if is_text_dtype(s):
        if looks_datetime(s):
            return "datetime_str"
        txt = s.dropna().astype(str)
        if len(txt):
            avg_len = float(txt.str.len().mean())
            avg_words = float(txt.str.count(r"\s+").mean()) + 1
            if avg_len > 40 or avg_words > 4:
                return "text"
        if nu == n_rows and n_rows > 20:
            return "id_like"
        if nu > max(50, 0.2 * n_rows):
            return "cat_high"
        return "cat_low" if nu <= 20 else "cat_high"
    return "other"


def detect_target(df, df_test, sub, id_guess):
    """Return (target, confidence, evidence)."""
    reserved = {c for c in ([id_guess] if id_guess else [])}
    if df_test is not None:
        diff = [c for c in df.columns if c not in df_test.columns and c not in reserved]
        if len(diff) == 1:
            return diff[0], "high", "the only column in train but not in test"
        if len(diff) > 1:
            if sub is not None:
                inter = [c for c in diff if c in sub.columns]
                if len(inter) == 1:
                    return inter[0], "high", ("in train, absent from test, and present "
                                              "in sample_submission")
                if len(inter) > 1:
                    return inter, "high", "multiple train-only cols also in submission"
            return diff[0], "medium", (f"first of {len(diff)} train-only columns "
                                       f"({diff})")
    if sub is not None:
        cands = [c for c in sub.columns if c in df.columns and c not in reserved]
        cands = [c for c in cands if c != (id_guess or "")]
        subonly = [c for c in sub.columns if c.lower() not in
                   {str(id_guess).lower(), "id"} and c in df.columns]
        if len(cands) == 1:
            return cands[0], "high", "the non-id sample_submission column"
        if len(subonly) == 1:
            return subonly[0], "medium", "derived from sample_submission columns"
    lowered = {str(c).lower(): c for c in df.columns}
    for hint in TARGET_NAME_HINTS:
        for lc, orig in lowered.items():
            if lc == hint and orig not in reserved:
                return orig, "medium", f"column name matches the common target '{hint}'"
    for hint in TARGET_NAME_HINTS:
        for lc, orig in lowered.items():
            if hint in lc and orig not in reserved:
                return orig, "low", f"column name contains '{hint}'"
    last = df.columns[-1]
    return last, "very low", "fell back to the LAST column — verify this before training"


def id_capable(s):
    """Could this column be an id? IDs are labels: integers or strings, never floats.

    Without this rule any continuous measurement that happens to be unique per row
    (e.g. a lognormal 'num_skewed') outranks the real id simply by being unique.
    """
    if is_text_dtype(s):
        return True
    if is_num_dtype(s):
        v = to_num(s)
        v = v[np.isfinite(v)]
        return bool(len(v)) and bool(np.allclose(v, np.round(v)))
    return False


def detect_id(df, df_test, sub):
    """Prefer a column that is unique in train, present in test, and named like an id."""
    n = len(df)
    scored = []
    for c in df.columns:
        s = df[c]
        if int(s.nunique(dropna=True)) != n or s.isna().any() or not id_capable(s):
            continue
        lname = str(c).lower()
        score = 0
        if sub is not None and c in sub.columns:
            score += 4
        if df_test is not None and c in df_test.columns:
            score += 2
        if lname in ID_EXACT_NAMES:
            score += 3
        elif name_tokens(c) & set(ID_TOKENS):
            score += 1
        if score:
            scored.append((score, -len(str(c)), str(c)))
    if scored:
        scored.sort(reverse=True)
        return scored[0][2]
    # No train-side id. The submission's first column still identifies the test rows,
    # which is exactly what the generated code needs to write submission.csv.
    if sub is not None and len(sub.columns):
        first = sub.columns[0]
        if first in df.columns or (df_test is not None and first in df_test.columns):
            return first
    return None


def detect_task(y, forced=None):
    """Return (task, class_map, evidence)."""
    y = y.dropna()
    nu = int(y.nunique())
    if forced:
        task = forced
    elif is_num_dtype(y):
        as_f = to_num(y)
        integral = bool(np.all(np.isfinite(as_f)) and np.allclose(as_f, np.round(as_f)))
        if nu == 2:
            task = "binary"
        elif integral and nu <= 20 and float(np.nanmin(as_f)) >= 0:
            task = "multiclass"
        else:
            task = "regression"
    else:
        task = "binary" if nu == 2 else "multiclass"
    cmap = None
    if task in ("binary", "multiclass"):
        classes = sorted(y.unique().tolist(), key=lambda v: str(v))
        cmap = {str(c): i for i, c in enumerate(classes)}
    ev = f"target has {nu} unique values, dtype `{y.dtype}`"
    return task, cmap, ev


# --- target encoding ------------------------------------------------------------------
# Every per-class number in this report (recalls, class names, probabilities) must use
# ONE encoding, and it has to be the same `class_to_index` map the report hands to the
# chatbot. pd.factorize() numbers classes by order of APPEARANCE, which does not match
# CLASS_MAP (built from the SORTED class list), so mixing the two silently attaches each
# score to the wrong class label. Always go through these helpers.

def encode_target(y):
    """Class labels -> CLASS_MAP integer codes (-1 for missing/unknown labels)."""
    if CLASS_MAP:
        return pd.Series(y).map(lambda v: CLASS_MAP.get(str(v), -1)) \
            .astype("int64").to_numpy()
    return pd.factorize(y)[0]


def class_label(i):
    """Human-readable label for model class index i (inverse of CLASS_MAP)."""
    inv = {v: k for k, v in (CLASS_MAP or {}).items()}
    return str(inv.get(int(i), i))


def target_rate(y):
    """Numeric target for 'mean target by group' tables.

    Regression -> the raw value. Classification -> a 0/1 indicator of REF_CLASS, so the
    tables read as "rate of class X" instead of "rate of whichever class came first".
    """
    if REF_CLASS is None:
        return to_num(y)
    return (pd.Series(y).astype(str) == str(REF_CLASS)).to_numpy(dtype="float64")


with section("schema", "3. SCHEMA INFERENCE"):
    need("DF")
    n = len(DF)

    ID_NAME = ID_COL or detect_id(DF, DF_TEST, SUB)
    say(f"- **ID column**: `{ID_NAME}`"
        + ("" if ID_COL else " _(auto-detected)_" if ID_NAME else " — none found"))

    if TARGET:
        tgt, conf, ev = (TARGET, "config", "set explicitly in CONFIG")
    else:
        tgt, conf, ev = detect_target(DF, DF_TEST, SUB, ID_NAME)
    TARGET_LIST = list(tgt) if isinstance(tgt, (list, tuple)) else [tgt]
    TARGET_LIST = [c for c in TARGET_LIST if c in DF.columns]
    if not TARGET_LIST:
        raise RuntimeError(f"target detection produced nothing usable (got {tgt!r})")
    TARGET_NAME = TARGET_LIST[0]
    say(f"- **TARGET**: `{TARGET_NAME}`  (confidence: **{conf}**; {ev})")
    if len(TARGET_LIST) > 1:
        say(f"  - multi-target task; other targets: `{TARGET_LIST[1:]}`. "
            f"This report profiles `{TARGET_NAME}`; treat the rest the same way.")
        flag("medium", f"Multi-target problem ({len(TARGET_LIST)} targets: "
                       f"{TARGET_LIST}); generated code must predict all of them.")
    if conf in ("low", "very low"):
        flag("high", f"Target was guessed with {conf} confidence as '{TARGET_NAME}' — "
                     f"confirm against the contest description before training.")

    y_full = DF[TARGET_NAME]
    TASK_NAME, CLASS_MAP, task_ev = detect_task(y_full, TASK)
    say(f"- **TASK**: **{TASK_NAME}**  ({'from CONFIG' if TASK else task_ev})")

    METRIC_GUESS = {"binary": "ROC-AUC (check: could be logloss / F1 / accuracy)",
                    "multiclass": "multiclass logloss or macro-F1 (check the rules)",
                    "regression": "RMSE (check: could be MAE / RMSLE / R2)"}[TASK_NAME]
    say(f"- **Metric guess**: {METRIC_GUESS}")
    say("  - The contest description overrides this guess; implement the stated metric.")

    if CLASS_MAP:
        say(f"- **Classes ({len(CLASS_MAP)})** and the index mapping to use:")
        if len(CLASS_MAP) <= 60:
            say("  `" + json.dumps(CLASS_MAP) + "`")
        else:
            head = dict(list(CLASS_MAP.items())[:60])
            say("  `" + json.dumps(head) + f"`  ... +{len(CLASS_MAP) - 60} more classes")
        # Reference class for every "mean target by X" table below. For binary this is
        # class index 1 -- the same column predict_proba()[:, 1] returns, so the report's
        # rates and its AUC talk about the same event.
        inv = {v: k for k, v in CLASS_MAP.items()}
        REF_CLASS = inv[1] if (TASK_NAME == "binary" and 1 in inv) else inv[0]
        say(f"- **Positive / reference class**: `{REF_CLASS}` (index "
            f"{CLASS_MAP[str(REF_CLASS)]}). Every 'target rate' table below is the rate "
            f"of this class, and binary AUC/probabilities refer to it.")

    # per-column type inference
    feature_cols = [c for c in DF.columns
                    if c not in TARGET_LIST and c != ID_NAME]
    for c in DF.columns:
        COLTYPES[c] = infer_type(DF[c], n, c)
    if ID_NAME and ID_NAME in COLTYPES:          # the id may live only in test/submission
        COLTYPES[ID_NAME] = "id_like"
    for t in TARGET_LIST:
        COLTYPES[t] = "target"

    groups = {}
    for c in feature_cols:
        groups.setdefault(COLTYPES[c], []).append(c)
    say("")
    say("**Inferred column types** (feature columns only):")
    say_table(["type", "count", "columns"],
              [[t, len(cs), cap_items(cs, 40)] for t, cs in sorted(groups.items())])

    for t, msg, sev in [
        ("constant", "constant (zero variance) — drop them", "medium"),
        ("id_like", "id-like (unique per row) — drop them as features, they only "
                    "memorize", "medium"),
        ("text", "free-text — need TF-IDF/embeddings or must be dropped", "medium"),
        ("datetime_str", "date-like strings — parse with pd.to_datetime and expand "
                         "into calendar features", "medium"),
    ]:
        if groups.get(t):
            flag(sev, f"{len(groups[t])} {msg}: {cap_items(groups[t], 20)}.")

    FACTS["schema"] = {
        "target": TARGET_NAME, "targets_all": TARGET_LIST,
        "target_confidence": conf, "id_col": ID_NAME,
        "task": TASK_NAME, "metric_guess": METRIC_GUESS,
        "class_map": CLASS_MAP, "positive_class": REF_CLASS,
        "column_types": {str(k): v for k, v in COLTYPES.items()},
        "feature_columns": list(map(str, feature_cols)),
    }

In [ ]:
# =====================================================================================
# 4. TARGET ANALYSIS
# =====================================================================================
with section("target", "4. TARGET ANALYSIS"):
    need("DF", "TARGET_NAME", "TASK_NAME")
    y = DF[TARGET_NAME]
    say(f"- dtype `{y.dtype}`, {int(y.notna().sum()):,} non-null, "
        f"{int(y.isna().sum()):,} null")
    if y.isna().any():
        flag("high", f"Target '{TARGET_NAME}' has {int(y.isna().sum()):,} missing "
                     f"values; drop or impute those rows before training.")

    if TASK_NAME == "regression":
        v = to_num(y)
        v = v[np.isfinite(v)]
        qs = [0, 1, 5, 25, 50, 75, 95, 99, 100]
        say("")
        say("**Target quantiles:**")
        say_table(["quantile", "value"],
                  [[f"p{q}", float(np.percentile(v, q))] for q in qs])
        mean, std = float(np.mean(v)), float(np.std(v))
        from scipy import stats as sps
        skew = float(sps.skew(v)) if len(v) > 2 else 0.0
        kurt = float(sps.kurtosis(v)) if len(v) > 3 else 0.0
        n_zero = int((v == 0).sum())
        n_neg = int((v < 0).sum())
        iqr = float(np.percentile(v, 75) - np.percentile(v, 25))
        lo, hi = np.percentile(v, 25) - 1.5 * iqr, np.percentile(v, 75) + 1.5 * iqr
        out_share = float(((v < lo) | (v > hi)).mean())
        say("")
        say_table(["stat", "value"], [
            ["mean", mean], ["std", std], ["skew", skew], ["excess kurtosis", kurt],
            ["zeros", f"{n_zero:,} ({100 * n_zero / len(v):.2f}%)"],
            ["negatives", f"{n_neg:,} ({100 * n_neg / len(v):.2f}%)"],
            ["IQR-outlier share", f"{100 * out_share:.2f}%"],
        ])
        if skew > 1.0 and n_neg == 0:
            say("")
            say("- **Right-skewed and non-negative** -> train on `np.log1p(y)` and "
                "invert with `np.expm1(pred)`. Section 12 reports both variants.")
            flag("high", f"Regression target is right-skewed (skew={skew:.2f}) and "
                         f"non-negative: train on log1p(y) and invert predictions, and "
                         f"consider RMSLE.")
        if out_share > 0.05:
            flag("medium", f"{100 * out_share:.1f}% of target values are IQR outliers; "
                           f"consider a robust/Huber objective or clipping predictions.")
        FACTS["target_stats"] = {"kind": "regression", "mean": mean, "std": std,
                                 "skew": skew, "zeros": n_zero, "negatives": n_neg,
                                 "min": float(v.min()), "max": float(v.max()),
                                 "p50": float(np.median(v)),
                                 "log1p_suggested": bool(skew > 1.0 and n_neg == 0)}
    else:
        vc = y.value_counts(dropna=False)
        say("")
        say("**Class distribution:**")
        say_table(["class", "count", "share_%"],
                  [[fmt(k), int(c), f"{100 * c / len(y):.3f}"] for k, c in vc.items()],
                  cap=60, unit="classes")
        imb = float(vc.max() / max(vc.min(), 1))
        say("")
        say(f"- majority **{fmt(vc.index[0])}** ({100 * vc.iloc[0] / len(y):.2f}%), "
            f"minority **{fmt(vc.index[-1])}** ({100 * vc.iloc[-1] / len(y):.3f}%), "
            f"imbalance ratio **{imb:.1f}x**")
        say(f"- accuracy of always predicting the majority class: "
            f"**{100 * vc.iloc[0] / len(y):.2f}%** — any model must beat this")
        if imb >= 10:
            flag("high", f"Severe class imbalance ({imb:.0f}x). Use "
                         f"StratifiedKFold, class weights or scale_pos_weight, and judge "
                         f"with AUC/PR-AUC/macro-F1 rather than accuracy.")
        elif imb >= 3:
            flag("medium", f"Class imbalance {imb:.1f}x: stratify the folds and prefer "
                           f"AUC/macro-F1 over accuracy.")
        tiny = vc[vc < 10]
        if len(tiny):
            flag("high", f"{len(tiny)} class(es) have <10 rows "
                         f"({cap_items(list(tiny.index), 15)}); stratified CV may fail — "
                         f"merge them, or reduce n_splits.")
        FACTS["target_stats"] = {
            "kind": "classification", "n_classes": int(y.nunique()),
            "imbalance_ratio": imb,
            "majority_class": str(vc.index[0]),
            "majority_share": float(vc.iloc[0] / len(y)),
            "class_counts": {str(k): int(v) for k, v in list(vc.items())[:200]},
        }

    # row-order dependence: a sorted file breaks naive KFold.
    # This is the one place order-of-appearance coding is the RIGHT choice: a file
    # grouped by class then yields monotonically increasing codes whatever the labels
    # sort to, which is exactly the pattern we want to catch.
    if TASK_NAME != "regression":
        codes = pd.factorize(y)[0].astype(float)
    else:
        codes = to_num(y)
    idx = np.arange(len(codes), dtype=float)
    ok = np.isfinite(codes)
    if ok.sum() > 10:
        from scipy import stats as sps
        rho = float(sps.spearmanr(idx[ok], codes[ok]).statistic)
        say("")
        say(f"- Spearman(row order, target) = **{rho:+.4f}**")
        if abs(rho) > 0.1:
            say("  - The file appears **sorted by the target**.")
            flag("high", f"Rows are ordered with respect to the target "
                         f"(Spearman {rho:+.2f}); always shuffle before splitting and "
                         f"never use a head/tail holdout.")
        FACTS.setdefault("target_stats", {})["row_order_spearman"] = rho

In [ ]:
# =====================================================================================
# 5. MISSING VALUES
# =====================================================================================
with section("missing", "5. MISSING VALUES"):
    need("DF", "TARGET_NAME")
    n = len(DF)
    miss = DF.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if not len(miss):
        say("- No missing values anywhere in train.")
    else:
        say(f"- **{len(miss)}** of {DF.shape[1]} train columns contain nulls.")
        say("")
        say_table(["column", "type", "n_missing", "missing_%"],
                  [[c, COLTYPES.get(c, "?"), int(v), f"{100 * v / n:.2f}"]
                   for c, v in miss.items()], cap=60, unit="columns")
        heavy = miss[miss / n > 0.5]
        if len(heavy):
            flag("medium", f"{len(heavy)} column(s) are >50% missing "
                           f"({cap_items(list(heavy.index), 15)}); drop them or encode "
                           f"only the is-missing indicator.")

    # does missingness itself predict the target?
    if TASK_NAME in ("binary", "multiclass") and len(miss):
        say("")
        say(f"**Is-missing indicator vs target** (rate of class `{REF_CLASS}` by "
            f"missingness; a large gap means missingness is informative — keep the "
            f"indicator):")
        ybin = target_rate(DF[TARGET_NAME])
        rows = []
        for c in list(miss.index)[:30]:
            m = DF[c].isna().to_numpy()
            if m.all() or (~m).all():
                continue
            rows.append([c, f"{ybin[m].mean():.4f}", f"{ybin[~m].mean():.4f}",
                         f"{abs(ybin[m].mean() - ybin[~m].mean()):.4f}"])
        rows.sort(key=lambda r: -float(r[3]))
        say_table(["column", "target_rate_when_missing", "when_present", "abs_gap"],
                  rows, cap=30, unit="columns")
        if rows and float(rows[0][3]) > 0.05:
            flag("medium", f"Missingness is predictive (largest gap "
                           f"{rows[0][3]} on '{rows[0][0]}'); add explicit "
                           f"`col_isna` indicator features instead of silent imputation.")
    elif TASK_NAME == "regression" and len(miss):
        say("")
        say("**Is-missing indicator vs target** (mean target by missingness):")
        yv = to_num(DF[TARGET_NAME])
        rows = []
        for c in list(miss.index)[:30]:
            m = DF[c].isna().to_numpy()
            if m.all() or (~m).all():
                continue
            a, b = float(np.nanmean(yv[m])), float(np.nanmean(yv[~m]))
            rows.append([c, a, b, abs(a - b)])
        rows.sort(key=lambda r: -float(r[3]))
        say_table(["column", "target_mean_when_missing", "when_present", "abs_gap"],
                  rows, cap=30, unit="columns")

    # train vs test missingness
    if DF_TEST is not None:
        shared = [c for c in DF.columns if c in DF_TEST.columns]
        rows = []
        for c in shared:
            a = float(DF[c].isna().mean())
            b = float(DF_TEST[c].isna().mean())
            if abs(a - b) > 0.02:
                rows.append([c, f"{100 * a:.2f}", f"{100 * b:.2f}",
                             f"{100 * (b - a):+.2f}"])
        rows.sort(key=lambda r: -abs(float(r[3])))
        say("")
        say("**Columns whose missing-rate differs between train and test (>2pp):**")
        say_table(["column", "train_%", "test_%", "delta_pp"], rows, cap=40,
                  unit="columns")
        if rows:
            flag("medium", f"Missingness differs between train and test "
                           f"(worst: '{rows[0][0]}' {rows[0][3]}pp); impute using "
                           f"statistics computed on train only, and never on the union.")
        all_null_test = [c for c in shared if DF_TEST[c].isna().all()
                         and not DF[c].isna().all()]
        if all_null_test:
            flag("high", f"Column(s) {cap_items(all_null_test, 10)} are entirely null "
                         f"in test but populated in train — they are useless at "
                         f"inference time, so drop them.")
    FACTS["missing"] = {"columns_with_nulls": int(len(miss)),
                        "worst": {str(c): float(v / n) for c, v in
                                  list(miss.items())[:30]}}

In [ ]:
# =====================================================================================
# 6. NUMERIC FEATURES
# =====================================================================================
with section("numeric", "6. NUMERIC FEATURES"):
    need("DF_SAMPLE", "TARGET_NAME")
    from scipy import stats as sps

    num_cols = [c for c, t in COLTYPES.items()
                if t in ("numeric", "num_lowcard", "binary_num", "bool")
                and c in DF_SAMPLE.columns]
    if not num_cols:
        say("- No numeric feature columns.")
    else:
        say(f"- **{len(num_cols)}** numeric feature column(s).")
        yv = DF_SAMPLE[TARGET_NAME]
        if TASK_NAME == "regression":
            ynum = to_num(yv)
            assoc_name = "spearman_vs_target"
        elif TASK_NAME == "binary":
            # 0/1 against the reference class, so the sign of the correlation means
            # "higher x -> more REF_CLASS" instead of something order-dependent.
            ynum = target_rate(yv)
            assoc_name = "spearman_vs_target"
        else:
            ynum = encode_target(yv).astype(float)
            assoc_name = "eta_sq_vs_target"

        def eta_sq(x, g):
            """Share of variance in x explained by group labels g."""
            ok = np.isfinite(x)
            x, g = x[ok], g[ok]
            if len(x) < 10:
                return np.nan
            gm = x.mean()
            ss_t = float(((x - gm) ** 2).sum())
            if ss_t <= 0:
                return 0.0
            ss_b = 0.0
            for lev in np.unique(g):
                xi = x[g == lev]
                ss_b += len(xi) * (xi.mean() - gm) ** 2
            return float(ss_b / ss_t)

        rows, skewed, assoc_map = [], [], {}
        for c in num_cols:
            x = to_num(DF_SAMPLE[c])
            fin = x[np.isfinite(x)]
            if not len(fin):
                rows.append([c, 0, *["-"] * 8])
                continue
            q25, q75 = np.percentile(fin, [25, 75])
            iqr = q75 - q25
            outl = float((((fin < q25 - 1.5 * iqr) | (fin > q75 + 1.5 * iqr)).mean())
                         if iqr > 0 else 0.0)
            sk = float(sps.skew(fin)) if len(fin) > 2 else 0.0
            ok = np.isfinite(x) & np.isfinite(ynum)
            if ok.sum() > 10:
                a = (eta_sq(x, ynum) if assoc_name.startswith("eta")
                     else float(sps.spearmanr(x[ok], ynum[ok]).statistic))
            else:
                a = np.nan
            a = 0.0 if (a is None or not np.isfinite(a)) else a
            assoc_map[c] = float(a)
            rows.append([c, int(DF_SAMPLE[c].nunique(dropna=True)), float(fin.mean()),
                         float(fin.std()), float(fin.min()), float(np.median(fin)),
                         float(fin.max()), sk, f"{100 * float((fin == 0).mean()):.1f}",
                         f"{100 * outl:.1f}", a])
            if abs(sk) > 1.5:
                skewed.append((c, sk))

        rows.sort(key=lambda r: -abs(r[-1]) if isinstance(r[-1], float) else 0)
        say("")
        say(f"**Numeric summary, sorted by |association with target| "
            f"({assoc_name}):**")
        if TASK_NAME == "binary":
            say(f"_Association is the Spearman correlation with P(target == "
                f"`{REF_CLASS}`), so a positive value means larger values of the column "
                f"go with that class._")
        say_table(["column", "nunique", "mean", "std", "min", "p50", "max", "skew",
                   "zero_%", "outlier_%", assoc_name], rows, cap=120, unit="columns")

        top = [r[0] for r in rows[:15]]
        say("")
        say(f"- Strongest numeric signals: `{cap_items(top, 15)}`")

        if skewed:
            skewed.sort(key=lambda t: -abs(t[1]))
            say(f"- Skewed (|skew|>1.5): "
                f"`{cap_items([f'{c} ({s:+.1f})' for c, s in skewed], 30)}`")
            flag("low", f"{len(skewed)} skewed numeric column(s); GBDTs tolerate skew, "
                        f"but log1p/rank-transform them if you add a linear or NN model.")

        # collinear pairs — only useful for linear models / feature pruning
        if len(num_cols) >= 2:
            sub = DF_SAMPLE[num_cols].apply(pd.to_numeric, errors="coerce")
            keep = [c for c in num_cols if sub[c].notna().sum() > 10
                    and sub[c].nunique() > 1]
            pairs = []
            if len(keep) >= 2:
                corr = sub[keep].corr(method="spearman").to_numpy()
                for i in range(len(keep)):
                    for j in range(i + 1, len(keep)):
                        r = corr[i, j]
                        if np.isfinite(r) and abs(r) > 0.95:
                            pairs.append([keep[i], keep[j], float(r)])
                pairs.sort(key=lambda p: -abs(p[2]))
            say("")
            say("**Near-duplicate numeric pairs (|Spearman| > 0.95):**")
            say_table(["col_a", "col_b", "spearman"], pairs, cap=40, unit="pairs")
            if pairs:
                flag("low", f"{len(pairs)} highly collinear numeric pair(s) "
                            f"(e.g. {pairs[0][0]} ~ {pairs[0][1]}); harmless for GBDTs, "
                            f"but drop one of each pair for linear models.")
        FACTS["numeric"] = {"n_columns": len(num_cols),
                            "top_associations": dict(sorted(
                                assoc_map.items(), key=lambda kv: -abs(kv[1]))[:30])}

In [ ]:
# =====================================================================================
# 7. CATEGORICAL FEATURES
# =====================================================================================
with section("categorical", "7. CATEGORICAL FEATURES"):
    need("DF_SAMPLE", "TARGET_NAME")
    cat_cols = [c for c, t in COLTYPES.items()
                if t in ("cat_low", "cat_high") and c in DF_SAMPLE.columns]
    if not cat_cols:
        say("- No categorical feature columns.")
    else:
        say(f"- **{len(cat_cols)}** categorical feature column(s).")
        n = len(DF_SAMPLE)

        def cramers_v(a, b):
            """Bias-corrected Cramer's V between two label arrays."""
            from scipy import stats as sps
            tab = pd.crosstab(a, b)
            if tab.size == 0 or min(tab.shape) < 2:
                return 0.0
            chi2 = float(sps.chi2_contingency(tab, correction=False)[0])
            nn = tab.to_numpy().sum()
            phi2 = chi2 / nn
            r, k = tab.shape
            phi2c = max(0.0, phi2 - (k - 1) * (r - 1) / max(nn - 1, 1))
            rc = r - (r - 1) ** 2 / max(nn - 1, 1)
            kc = k - (k - 1) ** 2 / max(nn - 1, 1)
            den = min(kc - 1, rc - 1)
            return float(np.sqrt(phi2c / den)) if den > 0 else 0.0

        yv = DF_SAMPLE[TARGET_NAME]
        rows = []
        for c in cat_cols:
            s = DF_SAMPLE[c]
            nu = int(s.nunique(dropna=True))
            vc = s.value_counts()
            rare = int((vc < 10).sum())
            unseen = "-"
            if DF_TEST is not None and c in DF_TEST.columns:
                tr_lv = set(DF[c].dropna().astype(str).unique())
                te = DF_TEST[c].dropna().astype(str)
                unseen = f"{100 * float((~te.isin(tr_lv)).mean()):.2f}" if len(te) else "-"
            try:
                if TASK_NAME == "regression":
                    grp = to_num(yv)
                    codes = pd.factorize(s.astype(str))[0]
                    gm = np.nanmean(grp)
                    ss_t = float(np.nansum((grp - gm) ** 2))
                    ss_b = sum(len(grp[codes == lv]) *
                               (np.nanmean(grp[codes == lv]) - gm) ** 2
                               for lv in np.unique(codes))
                    assoc = float(ss_b / ss_t) if ss_t > 0 else 0.0
                else:
                    assoc = cramers_v(s.astype(str).fillna("<NA>"), yv.astype(str))
            except Exception:
                assoc = 0.0
            rows.append([c, COLTYPES[c], nu, f"{100 * nu / max(n, 1):.2f}",
                         int(s.isna().sum()), rare, unseen, assoc,
                         cap_items([f"{k}:{v}" for k, v in vc.head(5).items()], 5)])
        rows.sort(key=lambda r: -(r[7] if isinstance(r[7], float) else 0))
        say("")
        say("**Categorical summary** (assoc = Cramer's V for classification, "
            "eta-squared for regression):")
        say_table(["column", "type", "nunique", "unique_%", "nulls", "levels_lt_10",
                   "unseen_in_test_%", "assoc_vs_target", "top_levels"],
                  rows, cap=80, unit="columns")

        hi = [r[0] for r in rows if r[1] == "cat_high"]
        if hi:
            flag("medium", f"High-cardinality categoricals ({cap_items(hi, 15)}): use "
                           f"frequency or out-of-fold target encoding, or LightGBM "
                           f"native categories — never plain one-hot.")
        bad_unseen = [(r[0], r[6]) for r in rows
                      if r[6] != "-" and float(r[6]) > 20]
        if bad_unseen:
            flag("high", f"Test contains many category levels unseen in train "
                         f"({', '.join(f'{c}: {v}%' for c, v in bad_unseen[:8])}); "
                         f"encode unseen levels to a reserved bucket, never drop rows.")

        # target mean by top levels, for the strongest few columns
        what = ("target mean" if TASK_NAME == "regression"
                else f"rate of class `{REF_CLASS}`")
        for c, *_ in rows[:5]:
            try:
                s = DF_SAMPLE[c].astype(str).fillna("<NA>")
                agg = target_rate(DF_SAMPLE[TARGET_NAME])
                g = pd.DataFrame({"lv": s, "y": agg}).groupby("lv")["y"]
                t = pd.DataFrame({"n": g.size(), "target_mean": g.mean()}) \
                    .sort_values("n", ascending=False)
                say("")
                say(f"**`{c}` — {what} by most frequent levels:**")
                say_table(["level", "n", "target_mean"],
                          [[i, int(r.n), float(r.target_mean)]
                           for i, r in t.head(20).iterrows()], cap=20, unit="levels")
            except Exception as exc:
                say(f"  (per-level table failed for {c}: {exc})")
        FACTS["categorical"] = {
            "n_columns": len(cat_cols),
            "high_cardinality": hi,
            "cardinalities": {str(r[0]): int(r[2]) for r in rows[:60]},
        }

In [ ]:
# =====================================================================================
# 8. DATETIME & STRUCTURE — time splits and group leakage
# =====================================================================================
TIME_SPLIT_DETECTED = False
TIME_COL_USED = None
GROUP_COL_USED = None

with section("time", "8. DATETIME & STRUCTURE"):
    need("DF")
    dt_cols = [c for c, t in COLTYPES.items()
               if t in ("datetime", "datetime_str") and c in DF.columns]
    if TIME_COL and TIME_COL in DF.columns and TIME_COL not in dt_cols:
        dt_cols.append(TIME_COL)

    if not dt_cols:
        say("- No datetime-like columns found.")
    else:
        say(f"- Datetime-like columns: `{cap_items(dt_cols, 30)}`")
        best = None
        for c in dt_cols:
            tr = parse_dt(DF[c])
            if tr.notna().sum() == 0:
                continue
            te = None
            if DF_TEST is not None and c in DF_TEST.columns:
                te = parse_dt(DF_TEST[c])
            say("")
            say(f"**`{c}`**")
            say_table(["split", "min", "max", "n_unique", "n_null"],
                      [["train", str(tr.min()), str(tr.max()),
                        int(tr.nunique()), int(tr.isna().sum())]]
                      + ([["test", str(te.min()), str(te.max()),
                           int(te.nunique()), int(te.isna().sum())]]
                         if te is not None and te.notna().sum() else []))
            if te is not None and te.notna().sum():
                later = bool(te.min() >= tr.max())
                overlap_days = (min(tr.max(), te.max()) - max(tr.min(), te.min())).days
                say(f"- test starts after train ends: **{later}** "
                    f"(overlap {overlap_days} days)")
                if later or overlap_days <= 0:
                    TIME_SPLIT_DETECTED = True
                    best = best or c
                    say("- **This is a time-based split.**")
            # target trend by month
            if TASK_NAME and TARGET_NAME in DF.columns and tr.notna().sum() > 30:
                try:
                    per = tr.dt.to_period("M").astype(str)
                    yv = target_rate(DF[TARGET_NAME])
                    g = pd.DataFrame({"m": per, "y": yv}).groupby("m")["y"]
                    t = pd.DataFrame({"n": g.size(), "target_mean": g.mean()})
                    say("")
                    say(f"**`{c}` — "
                        + ("target mean" if TASK_NAME == "regression"
                           else f"rate of class `{REF_CLASS}`") + " by month:**")
                    say_table(["month", "n", "target_mean"],
                              [[i, int(r.n), float(r.target_mean)]
                               for i, r in t.iterrows()], cap=40, unit="months")
                except Exception as exc:
                    say(f"  (monthly trend failed: {exc})")
        TIME_COL_USED = TIME_COL or best
        if TIME_SPLIT_DETECTED:
            flag("high", f"Test is a LATER time period than train (via '{TIME_COL_USED}'). "
                         f"Validate with an expanding-window / last-block split, never "
                         f"random KFold, and avoid features that leak the future.")

    # candidate group columns: repeated entities are a classic CV leak
    say("")
    say("**Candidate group columns** (repeated entities -> GroupKFold territory):")
    n = len(DF)
    cands = []
    for c in DF.columns:
        if c in TARGET_LIST or COLTYPES.get(c) in ("constant", "text"):
            continue
        nu = int(DF[c].nunique(dropna=True))
        if nu < 2 or nu > 0.5 * n:
            continue
        reps = n / nu
        if reps < 1.5:
            continue
        ov = "-"
        if DF_TEST is not None and c in DF_TEST.columns:
            tr_lv = set(DF[c].dropna().astype(str).unique())
            te_lv = set(DF_TEST[c].dropna().astype(str).unique())
            ov = (f"{100 * len(tr_lv & te_lv) / max(len(te_lv), 1):.1f}"
                  if te_lv else "-")
        cands.append([c, COLTYPES.get(c, "?"), nu, round(reps, 2), ov])
    cands.sort(key=lambda r: -r[3])
    say_table(["column", "type", "n_entities", "avg_rows_per_entity",
               "test_entities_seen_in_train_%"], cands, cap=40, unit="columns")

    GROUP_COL_USED = GROUP_COL
    if GROUP_COL_USED:
        say(f"- GROUP_COL from CONFIG: `{GROUP_COL_USED}` -> use GroupKFold.")
        flag("high", f"GROUP_COL='{GROUP_COL_USED}' is set: use GroupKFold so no entity "
                     f"appears in both train and validation folds.")
    elif cands:
        strong = [r for r in cands if r[3] >= 3 and r[1] in
                  ("cat_low", "cat_high", "num_lowcard", "binary_num")]
        if strong:
            say(f"- Strongest group candidate: `{strong[0][0]}` "
                f"({strong[0][3]} rows per entity on average).")
            flag("medium", f"'{strong[0][0]}' repeats across rows "
                           f"({strong[0][3]} rows/entity). If rows of one entity must "
                           f"not be split across folds, use GroupKFold on it; otherwise "
                           f"random KFold is fine.")
    else:
        say("- No strong repeated-entity column; plain KFold/StratifiedKFold is fine.")

    FACTS["structure"] = {"datetime_columns": dt_cols,
                          "time_split_detected": TIME_SPLIT_DETECTED,
                          "time_col": TIME_COL_USED,
                          "group_col": GROUP_COL_USED,
                          "group_candidates": [r[0] for r in cands[:10]]}

In [ ]:
# =====================================================================================
# 9. DUPLICATES & TRAIN/TEST OVERLAP
# =====================================================================================
with section("dupes", "9. DUPLICATES & OVERLAP"):
    need("DF", "TARGET_NAME")
    feat = [c for c in DF.columns if c not in TARGET_LIST and c != ID_NAME]
    hashable = [c for c in feat if COLTYPES.get(c) != "text"]

    full_dupes = int(DF.duplicated().sum())
    say(f"- Fully duplicated rows (all columns): **{full_dupes:,}** "
        f"({100 * full_dupes / max(len(DF), 1):.3f}%)")
    if full_dupes:
        flag("medium", f"{full_dupes:,} exactly duplicated train rows; de-duplicate "
                       f"before CV or identical rows will straddle folds and inflate "
                       f"your validation score.")

    if hashable:
        h = pd.util.hash_pandas_object(DF[hashable], index=False)
        dup_feat = int(h.duplicated().sum())
        say(f"- Rows with identical features (target ignored): **{dup_feat:,}**")
        if dup_feat:
            grp = pd.DataFrame({"h": h, "y": DF[TARGET_NAME].astype(str)})
            nun = grp.groupby("h")["y"].nunique()
            conflict = int((nun > 1).sum())
            say(f"- Feature-identical groups with **conflicting targets**: "
                f"**{conflict:,}**")
            if conflict:
                flag("high", f"{conflict:,} groups of feature-identical rows carry "
                             f"different targets — this is an irreducible noise floor. "
                             f"Do not chase a perfect score; consider averaging their "
                             f"targets.")
        # test rows that exactly match a train row
        if DF_TEST is not None:
            shared = [c for c in hashable if c in DF_TEST.columns]
            if shared:
                ht = pd.util.hash_pandas_object(DF_TEST[shared], index=False)
                htr = set(pd.util.hash_pandas_object(DF[shared], index=False).tolist())
                n_match = int(sum(1 for v in ht.tolist() if v in htr))
                say(f"- Test rows whose features exactly match a train row: "
                    f"**{n_match:,}** / {len(DF_TEST):,} "
                    f"({100 * n_match / max(len(DF_TEST), 1):.2f}%)")
                if n_match:
                    flag("high", f"{n_match:,} test rows are feature-identical to train "
                                 f"rows — copy the known train target straight onto "
                                 f"those predictions for free accuracy.")
                FACTS.setdefault("duplicates", {})["test_rows_matching_train"] = n_match

    if ID_NAME and ID_NAME in DF.columns:
        d = int(DF[ID_NAME].duplicated().sum())
        say(f"- Duplicated values in id column `{ID_NAME}`: **{d:,}**")
        if d:
            flag("medium", f"ID column '{ID_NAME}' is not unique ({d:,} repeats); it is "
                           f"a group key, not a row id — consider GroupKFold on it.")
    FACTS.setdefault("duplicates", {}).update({"full_duplicate_rows": full_dupes})

In [ ]:
# =====================================================================================
# HEAVY STAGES (10-12) — run concurrently so the CPU and the GPU both stay busy.
# Each stage returns its own lines/flags/facts, which are emitted in a fixed order
# afterwards, so output stays deterministic regardless of thread finish order.
# =====================================================================================
ADV_RESULT = LEAK_RESULT = BASE_RESULT = None
BASELINE_SUMMARY = None
CV_SCHEME = None


def build_matrix(df, cols, freq_maps=None, fit=False):
    """Numeric design matrix: calendar expansion + frequency encoding for high-card cats.

    `freq_maps` is fitted on train and reused for test so the encoding never leaks.
    """
    out = {}
    maps = freq_maps if freq_maps is not None else {}
    for c in cols:
        t = COLTYPES.get(c, "numeric")
        if c not in df.columns:
            continue
        s = df[c]
        if t in ("numeric", "num_lowcard", "binary_num"):
            out[c] = to_num(s)
        elif t == "bool":
            out[c] = to_num(s)
        elif t in ("datetime", "datetime_str"):
            d = parse_dt(s)
            out[f"{c}__year"] = to_num(d.dt.year)
            out[f"{c}__month"] = to_num(d.dt.month)
            out[f"{c}__day"] = to_num(d.dt.day)
            out[f"{c}__dow"] = to_num(d.dt.dayofweek)
            out[f"{c}__doy"] = to_num(d.dt.dayofyear)
            # Days since epoch. NOT via astype("int64"): that turns NaT into the int64
            # minimum (-9.2e18), which becomes a nonsense -1.1e8 "date" instead of NaN.
            out[f"{c}__epoch"] = ((d - pd.Timestamp("1970-01-01"))
                                  / pd.Timedelta(days=1)).to_numpy(dtype="float64")
        elif t == "cat_low":
            codes = s.astype(str)
            if fit:
                maps[c] = {v: i for i, v in enumerate(sorted(codes.dropna().unique()))}
            m = maps.get(c, {})
            out[c] = codes.map(lambda v: m.get(v, -1)).to_numpy(dtype="float64")
        elif t == "cat_high":
            codes = s.astype(str)
            if fit:
                maps[c] = codes.value_counts(normalize=True).to_dict()
            m = maps.get(c, {})
            out[f"{c}__freq"] = codes.map(lambda v: m.get(v, 0.0)) \
                .to_numpy(dtype="float64")
        # 'text', 'id_like', 'constant' are intentionally excluded
    X = pd.DataFrame(out, index=df.index)
    X = X.replace([np.inf, -np.inf], np.nan)
    return (X, maps) if fit else X


def model_cols():
    """Feature columns that are legitimate model inputs."""
    drop_types = {"constant", "id_like", "text", "target", "other"}
    return [c for c in DF.columns
            if c not in TARGET_LIST and c != ID_NAME
            and COLTYPES.get(c) not in drop_types]


def cv_splitter():
    """Pick the CV scheme the report recommends.

    Returns (splitter, name, reason, group_col_name). The 4th element is the group
    COLUMN NAME, never a prebuilt array: each caller splits its own row subset, so it
    has to build the group vector from that same subset or the groups silently
    misalign with X.
    """
    from sklearn.model_selection import (GroupKFold, KFold, StratifiedKFold,
                                         TimeSeriesSplit)
    if GROUP_COL_USED and GROUP_COL_USED in DF.columns:
        n_groups = int(DF[GROUP_COL_USED].nunique(dropna=False))
        why = (f"GROUP_COL='{GROUP_COL_USED}' ({n_groups} entities) keeps every entity "
               f"inside a single fold")
        if TIME_SPLIT_DETECTED and TIME_COL_USED:
            why += (f"; test is ALSO a later period than train, so additionally sort by "
                    f"'{TIME_COL_USED}' and keep the final block as a holdout")
        # GroupKFold cannot make more folds than there are entities
        return (GroupKFold(n_splits=max(2, min(N_FOLDS, n_groups))), "GroupKFold",
                why, GROUP_COL_USED)
    if TIME_SPLIT_DETECTED and TIME_COL_USED:
        return (TimeSeriesSplit(n_splits=max(2, N_FOLDS - 1)), "TimeSeriesSplit",
                f"expanding window over '{TIME_COL_USED}' (sort the rows by it first) "
                f"because test is a later period than train", None)
    if TASK_NAME in ("binary", "multiclass"):
        return (StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED),
                "StratifiedKFold", "classification with shuffled stratified folds", None)
    return (KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED),
            "KFold", "regression with shuffled folds", None)


def stage_adversarial(n_jobs):
    """Train-vs-test shift: per-feature AUC plus a model-based AUC."""
    lines, flags_, facts = [], [], {}
    if not RUN_ADVERSARIAL or DF_TEST is None:
        lines.append("- Skipped (no test file, or RUN_ADVERSARIAL=False).")
        return lines, flags_, facts
    from scipy.stats import rankdata
    from sklearn.model_selection import StratifiedKFold
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score

    cols = [c for c in model_cols() if c in DF_TEST.columns]
    if not cols:
        lines.append("- No shared feature columns to compare.")
        return lines, flags_, facts
    cap = min(MAX_ROWS // 2, 60_000)
    a = DF[cols].sample(min(cap, len(DF)), random_state=SEED)
    b = DF_TEST[cols].sample(min(cap, len(DF_TEST)), random_state=SEED)
    Xa, maps = build_matrix(a, cols, fit=True)
    Xb = build_matrix(b, cols, freq_maps=maps)
    Xb = Xb.reindex(columns=Xa.columns)
    X = pd.concat([Xa, Xb], ignore_index=True)
    y = np.r_[np.zeros(len(Xa)), np.ones(len(Xb))]

    # univariate: rank-based AUC == Mann-Whitney U, so no model needed
    uni = []
    for c in X.columns:
        v = X[c].to_numpy(dtype="float64")
        if not np.isfinite(v).any():
            continue
        v = np.where(np.isfinite(v), v, np.nanmedian(v[np.isfinite(v)]) if
                     np.isfinite(v).any() else 0.0)
        r = rankdata(v)
        n1 = float(y.sum())
        n0 = float(len(y) - n1)
        if n0 == 0 or n1 == 0:
            continue
        auc = (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n0 * n1)
        uni.append([c, float(max(auc, 1 - auc))])
    uni.sort(key=lambda r: -r[1])
    lines.append("**Per-feature train-vs-test shift (univariate AUC, 0.5 = no shift):**")
    lines.append(md_table(["feature", "shift_auc"], uni, cap=60, unit="features"))

    model_auc = float("nan")
    try:
        X = X.fillna(X.median(numeric_only=True)).fillna(0.0)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
        aucs, imps = [], np.zeros(X.shape[1])
        for tr, va in skf.split(X, y):
            m = HistGradientBoostingClassifier(max_iter=120, random_state=SEED)
            m.fit(X.iloc[tr], y[tr])
            aucs.append(roc_auc_score(y[va], m.predict_proba(X.iloc[va])[:, 1]))
        model_auc = float(np.mean(aucs))
        lines.append("")
        lines.append(f"- **Model-based adversarial AUC: {model_auc:.4f}** "
                     f"(folds: {', '.join(f'{a:.4f}' for a in aucs)})")
    except Exception as exc:
        lines.append(f"- model-based adversarial check failed: "
                     f"`{type(exc).__name__}: {exc}`")

    worst = [u[0] for u in uni[:8]]
    if np.isfinite(model_auc) and model_auc > 0.70:
        flags_.append(("high", f"STRONG train/test distribution shift (adversarial AUC "
                               f"{model_auc:.3f}). Top drivers: {cap_items(worst, 8)}. "
                               f"Consider dropping/binning those features, adversarial "
                               f"sample weighting, or validating on the train rows that "
                               f"most resemble test."))
    elif np.isfinite(model_auc) and model_auc > 0.60:
        flags_.append(("medium", f"Mild train/test shift (adversarial AUC "
                                 f"{model_auc:.3f}); drivers: {cap_items(worst, 6)}. "
                                 f"Prefer robust features and check CV-vs-LB agreement."))
    else:
        lines.append("- Train and test look like the same distribution.")
    facts = {"adversarial_auc": model_auc,
             "top_shift_features": {u[0]: u[1] for u in uni[:20]}}
    return lines, flags_, facts


def stage_leak(n_jobs):
    """Single-feature shallow-tree CV: a near-perfect lone feature means leakage."""
    lines, flags_, facts = [], [], {}
    if not RUN_LEAK_CHECK:
        lines.append("- Skipped (RUN_LEAK_CHECK=False).")
        return lines, flags_, facts
    from sklearn.model_selection import cross_val_score
    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

    cols = model_cols()[:MAX_LEAK_FEATURES]
    if not cols:
        lines.append("- No candidate feature columns.")
        return lines, flags_, facts
    cap = min(MAX_ROWS, 50_000)
    d = DF.sample(min(cap, len(DF)), random_state=SEED)
    X = build_matrix(d, cols, fit=True)[0]
    y = d[TARGET_NAME]
    if TASK_NAME == "regression":
        yv = to_num(y)
        ok = np.isfinite(yv)
        scoring, baseline_txt = "r2", "R2 (0 = predicting the mean)"
    else:
        yv = encode_target(y)
        ok = yv >= 0
        scoring = ("roc_auc" if TASK_NAME == "binary" else "accuracy")
        baseline_txt = ("AUC (0.5 = random)" if TASK_NAME == "binary"
                        else "accuracy (vs majority share)")
    X, yv = X.loc[ok], yv[ok]

    rows = []
    for c in X.columns:
        v = X[c].to_numpy(dtype="float64")
        if not np.isfinite(v).any() or np.nanstd(v) == 0:
            continue
        xx = np.nan_to_num(v, nan=float(np.nanmedian(v))).reshape(-1, 1)
        try:
            est = (DecisionTreeRegressor(max_depth=3, random_state=SEED)
                   if TASK_NAME == "regression"
                   else DecisionTreeClassifier(max_depth=3, random_state=SEED))
            sc = cross_val_score(est, xx, yv, cv=3, scoring=scoring, n_jobs=n_jobs)
            rows.append([c, float(np.mean(sc))])
        except Exception:
            continue
    rows.sort(key=lambda r: -r[1])
    lines.append(f"**Single-feature predictive power** — depth-3 tree, 3-fold, "
                 f"metric = {baseline_txt}:")
    lines.append(md_table(["feature", scoring], rows, cap=60, unit="features"))

    thr = {"regression": 0.95, "binary": 0.98, "multiclass": 0.95}[TASK_NAME]
    if TASK_NAME == "multiclass":
        maj = float(pd.Series(yv).value_counts(normalize=True).max())
        lines.append("")
        lines.append(f"- majority-class accuracy to beat: **{maj:.4f}**")
    leaks = [r for r in rows if r[1] >= thr]
    if leaks:
        names = [r[0].replace("__freq", "") for r in leaks]
        flags_.append(("high", f"LEAKAGE SUSPECT: feature(s) {cap_items(names, 10)} "
                               f"alone reach {scoring}={leaks[0][1]:.4f} with a depth-3 "
                               f"tree. Drop them unless the contest clearly intends "
                               f"them to be used."))
        lines.append("")
        lines.append(f"- **Suspicious:** `{cap_items(names, 20)}`")
    else:
        lines.append("")
        lines.append("- No single feature is suspiciously predictive on its own.")
    facts = {"single_feature_scores": {r[0]: r[1] for r in rows[:30]},
             "leak_suspects": [r[0] for r in leaks], "leak_metric": scoring}
    return lines, flags_, facts


def stage_baseline(n_jobs, use_gpu):
    """Quick CV baseline — the score the generated solution has to beat."""
    lines, flags_, facts = [], [], {}
    if not RUN_BASELINE:
        lines.append("- Skipped (RUN_BASELINE=False).")
        return lines, flags_, facts
    from sklearn.metrics import (accuracy_score, f1_score, log_loss,
                                 mean_absolute_error, mean_squared_error, r2_score,
                                 roc_auc_score, recall_score)

    cols = model_cols()
    leak_cols = set((LEAK_RESULT or (None, None, {}))[2].get("leak_suspects", [])) \
        if LEAK_RESULT else set()
    d = DF if len(DF) <= MAX_ROWS else DF.sample(MAX_ROWS, random_state=SEED)
    X, _maps = build_matrix(d, cols, fit=True)
    y_raw = d[TARGET_NAME]
    if TASK_NAME == "regression":
        y = to_num(y_raw)
        ok = np.isfinite(y)
    else:
        y = pd.factorize(y_raw)[0]
        ok = y >= 0
    X, y = X.loc[ok].reset_index(drop=True), y[ok]
    lines.append(f"- design matrix: **{X.shape[0]:,} rows x {X.shape[1]} features** "
                 f"(from {len(cols)} source columns)")
    lines.append(f"- excluded by type (constant / id-like / text): "
                 f"`{cap_items([c for c in DF.columns if COLTYPES.get(c) in ('constant', 'id_like', 'text')], 20)}`")
    if leak_cols:
        lines.append(f"- NOTE: leak suspects were kept in this baseline for reference: "
                     f"`{cap_items(sorted(leak_cols), 10)}` (drop them in the solution)")

    splitter, scheme_name, scheme_why, group_col = cv_splitter()
    groups = None
    if group_col is not None and group_col in d.columns:
        groups = d[group_col].to_numpy()
    lines.append(f"- CV: **{scheme_name}** ({scheme_why})")

    # model choice: GPU -> XGBoost (CUDA wheels); CPU -> LightGBM; else sklearn
    lgb = opt_import("lightgbm")
    xgb = opt_import("xgboost")
    engine = None
    if use_gpu and xgb is not None:
        engine = "xgboost-gpu"
    elif lgb is not None:
        engine = "lightgbm-cpu"
    elif xgb is not None:
        engine = "xgboost-cpu"
    else:
        engine = "sklearn-hgb"
    lines.append(f"- engine: **{engine}** (n_jobs={n_jobs})")

    def make_model():
        if engine.startswith("xgboost"):
            common = dict(n_estimators=400, max_depth=6, learning_rate=0.06,
                          subsample=0.9, colsample_bytree=0.9, random_state=SEED,
                          tree_method="hist", n_jobs=n_jobs,
                          device=("cuda" if use_gpu else "cpu"))
            if TASK_NAME == "regression":
                return xgb.XGBRegressor(**common)
            if TASK_NAME == "binary":
                return xgb.XGBClassifier(eval_metric="logloss", **common)
            return xgb.XGBClassifier(objective="multi:softprob",
                                     num_class=int(len(np.unique(y))),
                                     eval_metric="mlogloss", **common)
        if engine.startswith("lightgbm"):
            common = dict(n_estimators=400, learning_rate=0.06, num_leaves=31,
                          subsample=0.9, colsample_bytree=0.9, random_state=SEED,
                          n_jobs=n_jobs, verbose=-1)
            return (lgb.LGBMRegressor(**common) if TASK_NAME == "regression"
                    else lgb.LGBMClassifier(**common))
        from sklearn.ensemble import (HistGradientBoostingClassifier,
                                      HistGradientBoostingRegressor)
        common = dict(max_iter=300, learning_rate=0.08, random_state=SEED)
        return (HistGradientBoostingRegressor(**common) if TASK_NAME == "regression"
                else HistGradientBoostingClassifier(**common))

    def run_cv(y_target, label):
        """One full CV pass; returns (per-fold rows, oof predictions)."""
        oof = np.full(len(y_target), np.nan, dtype="float64")
        oof_proba = None
        rows, t0 = [], time.perf_counter()
        if groups is not None:
            sp = splitter.split(X, y_target, groups)
        elif TASK_NAME != "regression" or scheme_name == "TimeSeriesSplit":
            sp = splitter.split(X, y_target)
        else:
            sp = splitter.split(X)
        for k, (tr, va) in enumerate(sp):
            if time.perf_counter() - t0 > BASELINE_TIME_BUDGET_S:
                rows.append([f"fold {k}", "SKIPPED (time budget)", "-", "-"])
                break
            m = make_model()
            m.fit(X.iloc[tr], y_target[tr])
            if TASK_NAME == "regression":
                p = m.predict(X.iloc[va])
                oof[va] = p
                sc = float(np.sqrt(mean_squared_error(y_target[va], p)))
                rows.append([f"fold {k}", sc, len(tr), len(va)])
            else:
                pp = m.predict_proba(X.iloc[va])
                if oof_proba is None:
                    oof_proba = np.full((len(y_target), pp.shape[1]), np.nan)
                oof_proba[va] = pp
                if TASK_NAME == "binary":
                    sc = float(roc_auc_score(y_target[va], pp[:, 1]))
                else:
                    sc = float(f1_score(y_target[va], pp.argmax(1), average="macro"))
                rows.append([f"fold {k}", sc, len(tr), len(va)])
        return rows, oof, oof_proba

    rows, oof, oof_proba = run_cv(y, "raw")
    done = [r for r in rows if isinstance(r[1], float)]
    lines.append("")
    lines.append(f"**Per-fold scores** "
                 f"({'RMSE' if TASK_NAME == 'regression' else ('AUC' if TASK_NAME == 'binary' else 'macro-F1')}):")
    lines.append(md_table(["fold", "score", "n_train", "n_valid"], rows))
    if done:
        vals = [r[1] for r in done]
        lines.append("")
        lines.append(f"- **mean {np.mean(vals):.6f}  std {np.std(vals):.6f}** "
                     f"over {len(done)} fold(s)")

    summary = {"engine": engine, "cv_scheme": scheme_name, "n_folds_done": len(done),
               "fold_scores": [r[1] for r in done]}

    # full OOF metric suite
    if TASK_NAME == "regression":
        m = np.isfinite(oof) & np.isfinite(y)
        if m.sum() > 1:
            rmse = float(np.sqrt(mean_squared_error(y[m], oof[m])))
            mae = float(mean_absolute_error(y[m], oof[m]))
            r2 = float(r2_score(y[m], oof[m]))
            res = y[m] - oof[m]
            lines.append("")
            lines.append("**OOF metrics:**")
            tbl = [["RMSE", rmse], ["MAE", mae], ["R2", r2],
                   ["baseline RMSE (predict mean)",
                    float(np.sqrt(mean_squared_error(y[m], np.full(m.sum(), y[m].mean()))))]]
            if float(np.nanmin(y)) >= 0 and float(np.nanmin(oof[m])) > -1:
                try:
                    rmsle = float(np.sqrt(np.mean((np.log1p(np.clip(oof[m], 0, None))
                                                   - np.log1p(y[m])) ** 2)))
                    tbl.append(["RMSLE", rmsle])
                except Exception:
                    pass
            lines.append(md_table(["metric", "value"], tbl))
            lines.append("")
            lines.append("**Residual quantiles (y - pred):**")
            lines.append(md_table(["quantile", "residual"],
                                  [[f"p{q}", float(np.percentile(res, q))]
                                   for q in (1, 5, 25, 50, 75, 95, 99)]))
            summary.update({"metric": "RMSE", "score": rmse, "mae": mae, "r2": r2})

            # log1p variant for skewed non-negative targets
            if (FACTS.get("target_stats", {}) or {}).get("log1p_suggested"):
                try:
                    rows2, oof2, _ = run_cv(np.log1p(y), "log1p")
                    d2 = [r for r in rows2 if isinstance(r[1], float)]
                    mm = np.isfinite(oof2)
                    back = np.expm1(oof2[mm])
                    rmse_log = float(np.sqrt(mean_squared_error(y[mm], back)))
                    lines.append("")
                    lines.append(f"**log1p-target variant:** RMSE after inverting = "
                                 f"**{rmse_log:.6f}** vs raw **{rmse:.6f}** "
                                 f"-> {'log1p WINS' if rmse_log < rmse else 'raw wins'}")
                    summary["rmse_log1p"] = rmse_log
                    if rmse_log < rmse:
                        flags_.append(("high", f"Training on log1p(target) beats the raw "
                                               f"target ({rmse_log:.4f} vs {rmse:.4f} "
                                               f"RMSE); use log1p + expm1 in the "
                                               f"solution."))
                except Exception as exc:
                    lines.append(f"- log1p comparison failed: {exc}")
    else:
        if oof_proba is not None:
            m = np.isfinite(oof_proba).all(axis=1)
            if m.sum() > 1:
                yt, pp = y[m], oof_proba[m]
                pred = pp.argmax(1)
                tbl = [["accuracy", float(accuracy_score(yt, pred))],
                       ["macro-F1", float(f1_score(yt, pred, average="macro"))],
                       ["logloss", float(log_loss(yt, pp, labels=list(range(pp.shape[1]))))],
                       ["majority-class accuracy",
                        float(pd.Series(yt).value_counts(normalize=True).max())]]
                if TASK_NAME == "binary":
                    tbl.insert(0, ["ROC-AUC", float(roc_auc_score(yt, pp[:, 1]))])
                    summary.update({"metric": "ROC-AUC",
                                    "score": float(roc_auc_score(yt, pp[:, 1]))})
                else:
                    summary.update({"metric": "macro-F1",
                                    "score": float(f1_score(yt, pred, average="macro"))})
                lines.append("")
                lines.append("**OOF metrics:**")
                lines.append(md_table(["metric", "value"], tbl))

                rec = recall_score(yt, pred, average=None,
                                   labels=list(range(pp.shape[1])), zero_division=0)
                inv = {v: k for k, v in (CLASS_MAP or {}).items()}
                order = np.argsort(rec)
                lines.append("")
                lines.append("**Lowest per-class recall (the classes to fix):**")
                lines.append(md_table(
                    ["class_index", "class_label", "recall", "n_oof"],
                    [[int(i), str(sorted((CLASS_MAP or {}).keys())[i])
                      if CLASS_MAP and i < len(CLASS_MAP) else str(i),
                      float(rec[i]), int((yt == i).sum())] for i in order],
                    cap=30, unit="classes"))
                worst = [int(i) for i in order[:3]]
                if len(rec) and float(rec[order[0]]) < 0.5:
                    flags_.append(("medium", f"Class index {worst} have recall below "
                                             f"0.5 in the baseline; use class weights or "
                                             f"oversampling and report macro-F1."))

    # feature importances
    try:
        m = make_model()
        m.fit(X, y)
        imp = getattr(m, "feature_importances_", None)
        if imp is not None:
            order = np.argsort(imp)[::-1]
            lines.append("")
            lines.append("**Top feature importances (full-data fit):**")
            lines.append(md_table(["feature", "importance"],
                                  [[X.columns[i], float(imp[i])] for i in order],
                                  cap=40, unit="features"))
            summary["top_features"] = [str(X.columns[i]) for i in order[:20]]
    except Exception as exc:
        lines.append(f"- importance fit failed: `{type(exc).__name__}: {exc}`")

    facts = summary
    return lines, flags_, facts


with section("heavy", "10-12. SHIFT, LEAKAGE & BASELINE"):
    need("DF", "TARGET_NAME", "TASK_NAME")
    CV_SCHEME = cv_splitter()[1]
    nw = max(1, COMPUTE["cpu_workers"])
    use_gpu = bool(COMPUTE["use_gpu"])

    # The leak stage feeds the baseline's commentary, so it runs first (it is cheap).
    # Adversarial (CPU) then overlaps the baseline (GPU when chosen) so neither starves.
    say(f"- running stages with {nw} CPU worker(s); "
        f"baseline device = **{'GPU' if use_gpu else 'CPU'}**")
    LEAK_RESULT = stage_leak(n_jobs=min(nw, 4))

    if PARALLEL_HEAVY:
        try:
            # when the baseline owns the GPU, give the CPU stage most of the threads
            adv_jobs = max(1, nw - 2) if use_gpu else max(1, nw // 2)
            base_jobs = nw if use_gpu else max(1, nw - adv_jobs)
            with ThreadPoolExecutor(max_workers=2) as ex:
                fa = ex.submit(stage_adversarial, adv_jobs)
                fb = ex.submit(stage_baseline, base_jobs, use_gpu)
                ADV_RESULT, BASE_RESULT = fa.result(), fb.result()
            say("- adversarial + baseline ran concurrently (CPU and GPU overlapped)")
        except Exception as exc:
            say(f"- parallel execution failed ({type(exc).__name__}: {exc}); "
                f"falling back to sequential")
            ADV_RESULT = stage_adversarial(nw)
            BASE_RESULT = stage_baseline(nw, use_gpu)
    else:
        ADV_RESULT = stage_adversarial(nw)
        BASE_RESULT = stage_baseline(nw, use_gpu)

    # emit in a fixed order so the report is deterministic
    for title, res, fkey in [
        ("### 10. ADVERSARIAL VALIDATION (train vs test shift)", ADV_RESULT, "shift"),
        ("### 11. LEAKAGE SUSPECTS", LEAK_RESULT, "leakage"),
        ("### 12. QUICK CV BASELINE", BASE_RESULT, "baseline"),
    ]:
        say("")
        say(title)
        lines, flags_, facts = res if res else ([], [], {})
        for ln in lines:
            say(ln)
        for sev, txt in flags_:
            flag(sev, txt)
        FACTS[fkey] = facts
    BASELINE_SUMMARY = (BASE_RESULT or ([], [], {}))[2]
    gc.collect()

In [ ]:
# =====================================================================================
# 13. SUBMISSION CHECK
# =====================================================================================
with section("submission", "13. SUBMISSION CHECK"):
    if SUB is None:
        say("- No sample submission available; nothing to verify.")
    else:
        say(f"- path: `{SUB_PATH_USED}`")
        say(f"- shape: **{SUB.shape[0]:,} rows x {SUB.shape[1]} cols**")
        say(f"- columns and dtypes:")
        say_table(["column", "dtype", "example", "nunique"],
                  [[c, str(SUB[c].dtype), fmt(SUB[c].iloc[0]) if len(SUB) else "-",
                    int(SUB[c].nunique())] for c in SUB.columns])
        say("")
        say(f"- first ids: `{cap_items([fmt(v) for v in SUB.iloc[:5, 0]], 5)}`")
        say(f"- last ids:  `{cap_items([fmt(v) for v in SUB.iloc[-5:, 0]], 5)}`")

        if DF_TEST is not None:
            same = SUB.shape[0] == DF_TEST.shape[0]
            say("")
            say(f"- rows match test ({SUB.shape[0]:,} vs {DF_TEST.shape[0]:,}): "
                f"**{same}**")
            if not same:
                flag("high", f"sample_submission has {SUB.shape[0]:,} rows but test has "
                             f"{DF_TEST.shape[0]:,}; the submission may need aggregation "
                             f"or expansion — re-read the contest rules.")
            sub_id = SUB.columns[0]
            cand = ID_NAME if (ID_NAME and ID_NAME in DF_TEST.columns) else (
                sub_id if sub_id in DF_TEST.columns else None)
            if cand:
                a = SUB[sub_id].astype(str).to_numpy()
                b = DF_TEST[cand].astype(str).to_numpy()
                say(f"- id column `{sub_id}` vs test `{cand}`: "
                    f"same set **{set(a) == set(b)}**, "
                    f"same order **{len(a) == len(b) and bool((a == b).all())}**")
                if set(a) != set(b):
                    flag("high", f"Submission ids do not match test ids; build the "
                                 f"submission by merging on '{cand}', not by position.")
                elif not (len(a) == len(b) and bool((a == b).all())):
                    flag("medium", f"Submission id order differs from test row order; "
                                   f"reindex to the sample_submission order before "
                                   f"writing.")
            else:
                flag("medium", f"Could not match the submission id column "
                               f"'{sub_id}' to any test column; confirm how ids map.")
        pred_cols = [c for c in SUB.columns if c != SUB.columns[0]]
        say("")
        say(f"- **columns to predict**: `{pred_cols}`")
        for c in pred_cols:
            if is_num_dtype(SUB[c]):
                v = to_num(SUB[c])
                v = v[np.isfinite(v)]
                if len(v):
                    say(f"  - `{c}`: placeholder range [{v.min():g}, {v.max():g}], "
                        f"dtype `{SUB[c].dtype}`")
        say("")
        say("- Decide from the contest metric whether these must be **probabilities** "
            "or **hard labels** — AUC/logloss want probabilities, F1/accuracy want "
            "labels. Getting this wrong silently halves your score.")
        flag("medium", "Before submitting, assert: row count, id set, id order, dtype, "
                       "value range and zero NaNs against sample_submission.")
        FACTS["submission"] = {
            "path": SUB_PATH_USED, "columns": list(map(str, SUB.columns)),
            "id_column": str(SUB.columns[0]), "predict_columns": list(map(str, pred_cols)),
            "rows": int(SUB.shape[0]),
            "dtypes": {str(c): str(SUB[c].dtype) for c in SUB.columns},
        }

In [ ]:
# =====================================================================================
# 14. FIGURES
# =====================================================================================
with section("figures", "14. FIGURES"):
    if not RUN_FIGURES:
        say("- Skipped (RUN_FIGURES=False).")
    else:
        import matplotlib
        import matplotlib.pyplot as plt

        saved = []

        def finish(fig, name):
            p = OUT / name
            fig.tight_layout()
            fig.savefig(p, dpi=110, bbox_inches="tight")
            saved.append(str(p))
            try:
                plt.show()
            except Exception:
                pass
            plt.close(fig)

        # target distribution
        try:
            fig, ax = plt.subplots(figsize=(7, 4))
            if TASK_NAME == "regression":
                v = to_num(DF[TARGET_NAME])
                v = v[np.isfinite(v)]
                ax.hist(v, bins=60, color="#4C78A8")
                ax.set_title(f"target '{TARGET_NAME}' distribution")
            else:
                vc = DF[TARGET_NAME].value_counts().head(30)
                ax.bar([str(i) for i in vc.index], vc.to_numpy(), color="#4C78A8")
                ax.set_title(f"target '{TARGET_NAME}' class counts")
                ax.tick_params(axis="x", rotation=60)
            finish(fig, "fig_target.png")
        except Exception as exc:
            say(f"- target figure failed: {exc}")

        # missingness
        try:
            miss = DF.isna().mean()
            miss = miss[miss > 0].sort_values(ascending=False).head(30)
            if len(miss):
                fig, ax = plt.subplots(figsize=(7, max(3, 0.28 * len(miss))))
                ax.barh([str(i) for i in miss.index][::-1],
                        (100 * miss.to_numpy())[::-1], color="#E45756")
                ax.set_xlabel("% missing")
                ax.set_title("missingness (train, top 30)")
                finish(fig, "fig_missing.png")
        except Exception as exc:
            say(f"- missingness figure failed: {exc}")

        # train/test shift
        try:
            shift = ((FACTS.get("shift") or {}).get("top_shift_features") or {})
            if shift:
                items = list(shift.items())[:25][::-1]
                fig, ax = plt.subplots(figsize=(7, max(3, 0.3 * len(items))))
                ax.barh([k for k, _ in items], [v for _, v in items], color="#F58518")
                ax.axvline(0.5, ls="--", c="grey")
                ax.set_xlabel("univariate shift AUC (0.5 = no shift)")
                ax.set_title("train vs test shift by feature")
                finish(fig, "fig_shift.png")
        except Exception as exc:
            say(f"- shift figure failed: {exc}")

        # correlation heatmap of the most associated numeric features
        try:
            assoc = ((FACTS.get("numeric") or {}).get("top_associations") or {})
            top = [c for c in assoc if c in DF.columns][:15]
            if len(top) >= 2:
                sub = DF_SAMPLE[top].apply(pd.to_numeric, errors="coerce")
                corr = sub.corr(method="spearman").to_numpy()
                fig, ax = plt.subplots(figsize=(1 + 0.5 * len(top), 1 + 0.5 * len(top)))
                im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
                ax.set_xticks(range(len(top)))
                ax.set_xticklabels(top, rotation=90, fontsize=7)
                ax.set_yticks(range(len(top)))
                ax.set_yticklabels(top, fontsize=7)
                fig.colorbar(im, ax=ax, shrink=0.8)
                ax.set_title("Spearman correlation (top numeric)")
                finish(fig, "fig_corr.png")
        except Exception as exc:
            say(f"- correlation figure failed: {exc}")

        say(f"- saved {len(saved)} figure(s): `{cap_items([Path(s).name for s in saved], 10)}`")
        FACTS["figures"] = saved

In [ ]:
# =====================================================================================
# 15. CODE-WRITER FACTS — everything a chatbot needs, with no follow-up questions
# =====================================================================================
with section("codewriter", "15. CODE-WRITER FACTS"):
    need("DF", "TARGET_NAME", "TASK_NAME")
    g = {}
    for c in DF.columns:
        if c in TARGET_LIST or c == ID_NAME:
            continue
        g.setdefault(COLTYPES.get(c, "other"), []).append(str(c))

    numeric_feats = g.get("numeric", []) + g.get("num_lowcard", []) + \
        g.get("binary_num", []) + g.get("bool", [])
    cat_low = g.get("cat_low", [])
    cat_high = g.get("cat_high", [])
    dt_feats = g.get("datetime", []) + g.get("datetime_str", [])
    text_feats = g.get("text", [])
    drop_feats = g.get("constant", []) + g.get("id_like", []) + g.get("other", [])
    leak_suspects = (FACTS.get("leakage") or {}).get("leak_suspects", [])
    leak_clean = sorted({s.replace("__freq", "") for s in leak_suspects}
                        & set(map(str, DF.columns)))

    say("Paste this block verbatim; it is the contract the generated code must satisfy.")
    say("")
    say("```yaml")
    say(f"train_path: {Path(TRAIN_PATH).resolve()}")
    say(f"train_reader: {reader_call(TRAIN_PATH)}")
    say(f"test_path: {Path(TEST_PATH).resolve() if (TEST_PATH and Path(TEST_PATH).exists()) else 'null'}")
    say(f"test_reader: {reader_call(TEST_PATH) if TEST_PATH else 'null'}")
    say(f"sample_submission_path: {SUB_PATH_USED or 'null'}")
    say(f"n_train_rows: {len(DF)}")
    say(f"n_test_rows: {len(DF_TEST) if DF_TEST is not None else 'null'}")
    say("")
    say(f"target: {TARGET_NAME}")
    say(f"targets_all: {TARGET_LIST}")
    say(f"task: {TASK_NAME}")
    say(f"metric_guess: {METRIC_GUESS}")
    say(f"id_col: {ID_NAME or 'null'}")
    if CLASS_MAP:
        if len(CLASS_MAP) <= 40:
            say(f"class_to_index: {json.dumps(CLASS_MAP)}")
        else:
            say(f"class_to_index: <{len(CLASS_MAP)} classes — rebuild with "
                f"sorted(train['{TARGET_NAME}'].astype(str).unique())>")
    ts = FACTS.get("target_stats", {}) or {}
    say(f"target_transform: {'log1p (invert with expm1)' if ts.get('log1p_suggested') else 'none'}")
    say("")
    say("feature_columns:")
    say(f"  numeric: {numeric_feats}")
    say(f"  categorical_low_card: {cat_low}")
    say(f"  categorical_high_card: {cat_high}")
    say(f"  datetime: {dt_feats}")
    say(f"  text: {text_feats}")
    say(f"  drop_always: {drop_feats}")
    say(f"  drop_leak_suspects: {leak_clean}")
    say("")
    say("encoding_recipe:")
    say("  numeric: pass through; GBDTs need no scaling")
    say("  categorical_low_card: ordinal/category dtype (LightGBM native) or one-hot")
    say("  categorical_high_card: frequency encoding, or out-of-fold target encoding")
    say("  datetime: pd.to_datetime(..., format='mixed') then year/month/day/dow/doy/epoch")
    say("  unseen_categories: map to a reserved -1 bucket, never drop rows")
    say("")
    sch = cv_splitter()
    say(f"cv_scheme: {sch[1]}")
    say(f"cv_reason: {sch[2]}")
    say(f"cv_n_splits: {N_FOLDS if sch[1] != 'TimeSeriesSplit' else max(2, N_FOLDS - 1)}")
    say(f"cv_group_col: {GROUP_COL_USED or 'null'}")
    say(f"cv_time_col: {TIME_COL_USED or 'null'}")
    say(f"seed: {SEED}")
    say("")
    bs = BASELINE_SUMMARY or {}
    say(f"baseline_engine: {bs.get('engine', 'n/a')}")
    say(f"baseline_metric: {bs.get('metric', 'n/a')}")
    say(f"baseline_score_to_beat: {bs.get('score', 'n/a')}")
    say(f"baseline_fold_scores: {[round(float(s), 6) for s in bs.get('fold_scores', [])]}")
    say(f"baseline_top_features: {bs.get('top_features', [])}")
    say("")
    if SUB is not None:
        say(f"submission_columns: {list(map(str, SUB.columns))}")
        say(f"submission_id_column: {SUB.columns[0]}")
        say(f"submission_predict_columns: "
            f"{[str(c) for c in SUB.columns if c != SUB.columns[0]]}")
        say(f"submission_dtypes: "
            f"{json.dumps({str(c): str(SUB[c].dtype) for c in SUB.columns})}")
        say(f"submission_rows_expected: {len(SUB)}")
    else:
        say("submission_columns: unknown (no sample_submission found)")
    say("submission_output_filename: submission.csv")
    say("")
    say("compute:")
    say(f"  platform: {COMPUTE['platform']}")
    say(f"  cpu_cores: {COMPUTE['cpu_count']}")
    say(f"  gpus_present: {COMPUTE['n_gpus_present']} {COMPUTE['gpu_names']}")
    say(f"  gpus_usable: {COMPUTE['gpu_budget']}")
    say(f"  gpu_worth_it_for_this_data: {COMPUTE['use_gpu']}  # {COMPUTE['gpu_reason']}")
    say(f"  internet: {COMPUTE['internet']}")
    say(f"  installed_libs_missing: {COMPUTE['missing_libs']}")
    say("```")
    FACTS["code_writer"] = {
        "numeric": numeric_feats, "cat_low": cat_low, "cat_high": cat_high,
        "datetime": dt_feats, "text": text_feats, "drop_always": drop_feats,
        "drop_leak_suspects": leak_clean,
        "cv_scheme": sch[1], "cv_reason": sch[2],
        "target_transform": "log1p" if ts.get("log1p_suggested") else "none",
        "baseline": bs, "output_filename": "submission.csv",
    }

In [ ]:
# =====================================================================================
# FINAL — assemble the report, print it, save report.md + facts.json
# =====================================================================================
TOTAL_S = time.perf_counter() - T_START


# Everything below must survive ANY earlier failure: this is the cell that writes the
# deliverables. So it only reads globals through gget()/safe defaults, and every block
# that could still raise is individually wrapped.

def gget(name, default=None):
    """Read a global an earlier section may never have set."""
    v = globals().get(name, default)
    return default if v is None else v


def num_fmt(v):
    """Thousands-separated int, or '?' for anything non-numeric (e.g. a failed load)."""
    return f"{int(v):,}" if isinstance(v, (int, float, np.integer, np.floating)) else "?"


def build_summary_block():
    """Section 0: the most valuable part, so it comes first in the pasted report."""
    L = ["## 0. SUMMARY", ""]
    ts = FACTS.get("target_stats") or {}
    bs = gget("BASELINE_SUMMARY", {}) or {}
    data = FACTS.get("data") or {}
    df_test = gget("DF_TEST")
    L.append(f"- **Task**: {gget('TASK_NAME', 'UNKNOWN')} | "
             f"**target** `{gget('TARGET_NAME', 'UNKNOWN')}` | "
             f"**metric guess** {gget('METRIC_GUESS', 'unknown')}")
    L.append(f"- **Train** {num_fmt(data.get('train_rows'))} rows x "
             f"{data.get('train_cols', '?')} cols"
             + (f" | **Test** {len(df_test):,} rows" if df_test is not None
                else " | no test file"))
    L.append(f"- **Paths**: train `{data.get('train_path', TRAIN_PATH)}`"
             + (f" | test `{data.get('test_path')}`" if data.get("test_path") else "")
             + (f" | submission `{gget('SUB_PATH_USED')}`"
                if gget("SUB_PATH_USED") else ""))
    L.append(f"- **ID column**: `{gget('ID_NAME', 'none')}` | "
             f"**CV scheme**: {gget('CV_SCHEME', 'n/a')}")
    if ts.get("kind") == "classification":
        L.append(f"- **Classes**: {ts.get('n_classes')} | imbalance "
                 f"{float(ts.get('imbalance_ratio') or float('nan')):.1f}x | majority "
                 f"'{ts.get('majority_class')}' at "
                 f"{100 * float(ts.get('majority_share') or 0):.2f}%")
    elif ts.get("kind") == "regression":
        L.append(f"- **Target range** [{ts.get('min')}, {ts.get('max')}], "
                 f"median {ts.get('p50')}, skew {float(ts.get('skew') or 0):.2f}"
                 f"{' -> use log1p' if ts.get('log1p_suggested') else ''}")
    if isinstance(bs.get("score"), float) and math.isfinite(bs["score"]):
        L.append(f"- **Baseline to beat**: {bs.get('metric')} = "
                 f"**{bs['score']:.6f}** ({bs.get('engine')}, "
                 f"{bs.get('n_folds_done')} folds of {bs.get('cv_scheme')})")
    adv = (FACTS.get("shift") or {}).get("adversarial_auc")
    if isinstance(adv, float) and math.isfinite(adv):
        L.append(f"- **Train/test shift**: adversarial AUC {adv:.4f}")
    comp = gget("COMPUTE", {}) or {}
    L.append(f"- **Compute**: {comp.get('platform', '?')}, "
             f"{comp.get('cpu_count', '?')} CPU cores, "
             f"{comp.get('n_gpus_present', '?')} GPU(s) present / "
             f"{comp.get('gpu_budget', '?')} usable, "
             f"internet={comp.get('internet', '?')}")
    L.append(f"- **Profiler runtime**: {TOTAL_S:.1f}s")
    failed = [s["title"] for s in SECTIONS if s["error"]]
    L.append(f"- **Sections failed**: {len(failed)}"
             + (f" -> {failed}" if failed else " (all sections completed)"))
    if failed:
        L.append("- The failed sections' facts are missing; treat anything they would "
                 "have reported as UNKNOWN rather than assuming a default.")
    return L


def build_flags_block():
    L = ["## AUTO-FLAGS", "",
         "Each line is a problem plus the action the generated code must take.", ""]
    rank = {"high": 0, "medium": 1, "low": 2}
    items = sorted(FLAGS.items(), key=lambda kv: rank.get(kv[1], 3))
    if not items:
        L.append("- None. Nothing alarming was detected.")
    for i, (text, sev) in enumerate(items, 1):
        L.append(f"{i}. **[{sev.upper()}]** {text}")
    return L


def assemble_report():
    """Build the full report; degrade to a minimal one rather than ever raising."""
    head = ["# TABULAR DATASET PROFILE REPORT", "",
            f"Generated by `tabular_profiler.ipynb` in {TOTAL_S:.1f}s. "
            f"Seed {SEED}. pandas {pd.__version__}, numpy {np.__version__}.", ""]
    for builder in (build_summary_block, build_flags_block):
        try:
            head += builder() + [""]
        except Exception as exc:                      # must never lose the report
            head += [f"_(summary block failed: {type(exc).__name__}: {exc})_", ""]
    try:
        head += ["## SECTION TIMINGS", "",
                 md_table(["section", "seconds", "status"],
                          [[s["title"], round(s["seconds"], 2),
                            "FAILED" if s["error"] else "ok"] for s in SECTIONS]), ""]
    except Exception:
        pass
    for s in SECTIONS:
        head += [f"## {s['title']}", ""] + list(s["lines"]) + [""]
    head += ["", "---", "", "END OF REPORT"]
    return "\n".join(head)


try:
    REPORT_TEXT = assemble_report()
except Exception as exc:                              # last-resort minimal report
    REPORT_TEXT = ("# TABULAR DATASET PROFILE REPORT\n\n"
                   f"Report assembly failed: {type(exc).__name__}: {exc}\n\n"
                   + "\n".join(f"## {s['title']}\n\n" + "\n".join(s["lines"])
                               for s in SECTIONS)
                   + "\n\nEND OF REPORT")

rp = OUT / "report.md"
rp.write_text(REPORT_TEXT, encoding="utf-8")

FACTS["_meta"] = {"runtime_seconds": round(TOTAL_S, 2), "seed": SEED,
                  "n_flags": len(FLAGS),
                  "sections_failed": [s["title"] for s in SECTIONS if s["error"]],
                  "report_chars": len(REPORT_TEXT)}
FACTS["flags"] = [{"severity": v, "text": k} for k, v in FLAGS.items()]
fp = OUT / "facts.json"
try:
    fp.write_text(json.dumps(jsonable(FACTS), indent=2), encoding="utf-8")
except Exception as exc:                              # facts.json must always parse
    fp.write_text(json.dumps({"_meta": FACTS["_meta"],
                              "facts_error": f"{type(exc).__name__}: {exc}"}, indent=2),
                  encoding="utf-8")

print("\n" + "=" * 84)
print(f"COPY EVERYTHING BETWEEN THE LINES  ({len(REPORT_TEXT):,} characters)")
print("=" * 84 + "\n")
print(REPORT_TEXT)
print("\n" + "=" * 84)
print("END OF REPORT — COPY EVERYTHING BETWEEN THE LINES")
print("=" * 84)
print(f"\nsaved: {rp.resolve()}")
print(f"saved: {fp.resolve()}")
print(f"flags: {len(FLAGS)} | sections failed: {len(FACTS['_meta']['sections_failed'])} "
      f"| total runtime: {TOTAL_S:.1f}s")

# STAGE 2 — paste these into a chatbot

Four prompts, in order. Replace `<<<CONTEST DESCRIPTION>>>` with the competition page
text and `<<<REPORT>>>` with everything between the COPY markers printed above.

If the report is too long for one message, paste it in parts — it is ordered by
importance, so `0. SUMMARY` + `AUTO-FLAGS` + `15. CODE-WRITER FACTS` alone are enough
to get working code.

---

## Message 1 — write the full solution now

```text
You are writing a complete, runnable solution for a tabular machine-learning
competition. I will copy-paste your code and run it as-is.

OUTPUT RULES (follow exactly):
- Output only code. Reply with complete, runnable code blocks and nothing else except
  short code comments. No advice, no summaries, no bullet lists of tips.
- Never ask me questions. If something is ambiguous, pick the most reasonable option
  from the report and state the assumption in a one-line comment at the top of the code.
- Never give partial snippets or diffs. Always give the full cell(s) ready to paste, in
  execution order, with every import included.
- Read exact paths, column names, class names and the submission format from the
  report. Do not use placeholders for them.
- Code must run on Kaggle/Colab with a T4 and no internet unless the report says
  internet is available. Use only libraries the report lists as installed.
- Handle every AUTO-FLAG in the report inside the code (leakage columns dropped,
  group/time CV, imbalance handling, distribution shift, duplicates, etc.).

WHAT TO BUILD — one notebook as consecutive code cells:
1. CONFIG cell: all paths, seed, fold count, and a TIME_BUDGET_MIN variable; set the
   total budget so the whole run finishes in 30-40 minutes.
2. Data loading using the exact reader and paths from the report.
3. Feature engineering driven by the report's column lists: calendar expansion for
   datetime columns, frequency or out-of-fold target encoding for high-cardinality
   categoricals, explicit is-missing indicators where the report says missingness is
   predictive. Drop every column the report lists under drop_always and
   drop_leak_suspects.
4. Validation scheme exactly as cv_scheme says in the report, so it matches the test
   distribution.
5. Model: a GBDT ensemble (LightGBM and/or XGBoost and/or CatBoost, whichever the
   report lists as installed), with AUTOMATIC HYPERPARAMETER TUNING - use Optuna if
   installed, otherwise a built-in randomized search over a sensible grid - with the
   tuning step bounded by its own time budget variable and the best params printed.
   Apply the target transform the report recommends.
6. Implement the exact competition metric as a function and use it for out-of-fold
   scoring. Print per-fold and overall OOF scores.
7. CPU/GPU usage: detect the device at runtime. Use the GPU for model training only
   when the report says gpu_worth_it_for_this_data is true or the frame is large;
   otherwise train on CPU. Keep both busy and never let either starve: run feature
   engineering, encoding and data prep on CPU worker threads while a GPU model trains,
   and if two GPUs are present (Kaggle) split folds or ensemble members across both
   devices. Set n_jobs for CPU models to the detected core count, and do not
   oversubscribe when a GPU model is already running.
8. Test inference plus a submission writer that matches the sample submission exactly,
   with built-in asserts: row count, id set, id order, dtype, value range, no NaN.

The first run must produce a valid submission file. Beat this baseline from the report:
baseline_score_to_beat.

CONTEST DESCRIPTION:
<<<CONTEST DESCRIPTION>>>

DATASET REPORT:
<<<REPORT>>>
```

---

## Message 2 — upgrade to the best version

```text
Here are the OOF/CV scores from your code:
<<<PASTE PER-FOLD AND OOF SCORES>>>

Rewrite the COMPLETE notebook (not a diff) adding the three highest-gain improvements
for the time I have left. Choose from: stronger/more diverse models, more folds or
seeds, blending and stacking with weights tuned on OOF, out-of-fold target encoding,
pseudo-labeling on confident test rows, deeper hyperparameter tuning, feature selection
driven by importance, and better handling of the shift the report reports.

Keep all Message 1 output rules. Keep automatic hyperparameter tuning. Give each added
step its own configurable time budget variable so I can cut it if I run short. Keep the
CPU/GPU overlap and the 2-GPU split. Still write a valid submission at the end, and
still print OOF scores for the exact competition metric. Code only.

CONTEST DESCRIPTION:
<<<CONTEST DESCRIPTION>>>

DATASET REPORT:
<<<REPORT>>>
```

---

## Message 3 — fix and iterate

```text
Your code failed, or the scores need work. Here is the evidence:

<<<PASTE THE FULL TRACEBACK, OR THE NEW SCORES PLUS ERROR-ANALYSIS NOTES:
   worst classes, biggest residuals, lowest-recall classes>>>

Give me the complete corrected cell(s), ready to paste, with the fix applied. Keep all
Message 1 output rules: code only, no explanation beyond code comments, no diffs, full
cells in execution order. If the error implies a wrong assumption about the data, fix
the assumption and note it in a one-line comment.
```

---

## Message 4 — final safety pass

```text
Output the full final notebook one more time with these guarantees added:
- A submission-validation block that asserts row count, id set, id order, dtype, value
  range and absence of NaN against the sample submission, and raises loudly on failure.
- All seeds fixed (python, numpy, and every model library used).
- A fallback path: wrap the main training in try/except and on any failure, or if the
  time budget is exhausted, train a cheap model (single LightGBM with default params,
  or a mean/majority predictor as the last resort) so that a valid submission file
  ALWAYS exists on disk when the notebook finishes.
- Write the submission file early with the fallback prediction, then overwrite it with
  the better prediction once training succeeds.

Code only, complete notebook, all Message 1 output rules still apply.
```